<div align="center" style="padding:28px 16px;border-radius:16px;background:linear-gradient(135deg,#6366f1,#8b5cf6 55%,#db2777);color:#fff">
<div style="font-size:34px;font-weight:800;letter-spacing:.5px">&#127908; DiffSinger Colab</div>
<div style="font-size:15px;opacity:.92;margin-top:6px">Singing voice synthesis via shallow diffusion &mdash; rebuilt, faster, and it tells you what went wrong</div>
<div style="margin-top:14px;font-size:13px">
<a style="color:#fff" href="https://github.com/openvpi/DiffSinger">DiffSinger</a> &nbsp;&middot;&nbsp;
<a style="color:#fff" href="https://github.com/xunmengshe/OpenUtau/wiki/Voicebank-Development">OpenUtau voicebank docs</a>
</div>
</div>

### &#129517; How to use it

| I want to&hellip; | Run |
|---|---|
| &#127926; Train an acoustic model | **0 &rarr; 1 &rarr; 2 &rarr; 3 &rarr; 4 &rarr; 5** |
| &#127932; Train a variance model | **0 &rarr; 1 &rarr; 2 &rarr; 3 &rarr; 4 &rarr; 5** &nbsp;<font size="-1">(pick `variance` in step 3)</font> |
| &#128260; Continue yesterday's run | **0 &rarr; 1 &rarr; 5** &nbsp;<font size="-1">(fill in `resume_from_model_dir`)</font> |
| &#128230; Only export ONNX / build a voicebank | **0 &rarr; 1** <font size="-1">(`profile = onnx export only`)</font> **&rarr; 6 &rarr; 7** |

### &#9889; Pick a training mode

Chosen in **step 3** (and can be changed again in **step 5**).

| | &#128994; Original | &#9889; Woah's Epic Faster Training <sup>experimental</sup> |
|---|---|---|
| Config | stock DiffSinger, exactly as before | stock DiffSinger keys, tuned to the runtime |
| Binarize | single process | one worker per CPU core (falls back to single-process if it fails) |
| Batch size | your sliders in step 5 | sized to the GPU's VRAM (T4 / L4 / A100 tiers) |
| Precision | what you picked in step 3 | **fp16 tensor cores** on a T4, **bf16** on L4 / A100 |
| Data loading | 4 workers, prefetch 2 | one core left free for the training loop, prefetch 4, expandable CUDA allocator |
| Progress bar | redrawn every step | every 10 steps (less CPU stolen from the GPU on 2-core runtimes) |
| Compiler | eager PyTorch | `torch.compile` on the diffusion backbone (auto-falls back to eager) |
| Optimizer | stock | fused AdamW kernel (Muon branch on any GPU, `main` under bf16) |
| Validation / logging | 10 rendered samples + sanity pass, log every 100 | 3 samples, no sanity pass, log every 500 |
| Binary data | local or Drive | always local disk |

Fast mode changes **how quickly** steps run, never the model you get: architecture, schedule,
learning rate and checkpoints are identical. It is tuned for the free **T4** first: that
card is limited by memory bandwidth and Colab gives it only 2 CPU cores, so the wins there come
from `torch.compile` fusing the backbone's many small kernels and from keeping the CPU free
to feed the GPU. On an L4 / A100, bf16 and bigger batches add on top. If
training dies with *CUDA out of memory*, switch step 5 back to **Original** &mdash; the cell
tells you when that is the cause.

### &#128161; Worth knowing

- Every step says what it did and how long it took. When something fails you get the problem,
  the relevant log lines, and what to do about it.
- State lives in `/content/ds_state.json`, not in Python variables, so a runtime restart costs
  you nothing.
- Nothing in the DiffSinger source tree is patched, so re-running any cell is safe.
- DiffSinger gets **its own Python 3.11 environment**: its `numpy<2` / `librosa<0.10` pins have
  no wheels on the Python 3.13 Colab now ships, and compiling them is what made the old setup
  take an hour.

<font size="-1">Terminology: what OpenUtau calls **parameters** (tension, voicing, breathiness,
energy) is what DiffSinger's configs call **variance**.</font>

---
## &#128295; Setup

Run cell **0** once, then cell **1**. Both are safe to re-run at any point.

### 0 &middot; Helper library
Writes `/content/dsnb.py`. Everything else imports it, which is why the notebook survives a runtime restart. Run it once.

In [ ]:
%%writefile /content/dsnb.py
"""
dsnb.py - shared helper library for the DiffSinger Colab notebook.

Written to /content so every other cell can `import dsnb` even after a runtime
restart.  This is why the notebook does not depend on variables left over in
memory from earlier cells - the single biggest source of "NameError" in the
original notebook.
"""

import contextlib
import json
import os
import re
import shlex
import shutil
import subprocess
import sys
import time
import wave
import zipfile
from pathlib import Path

CONTENT = Path("/content")
REPO = CONTENT / "DiffSinger"
STATE_PATH = CONTENT / "ds_state.json"
LOG_DIR = CONTENT / "nb_logs"

# DiffSinger pins numpy<2 and librosa<0.10, which means it needs Python 3.12 or
# older.  Colab now ships 3.13, where those pins have no wheels at all and pip
# falls back to compiling numpy, onnx and onnxsim from source on two cores - the
# single biggest reason setup "takes an hour and then fails".  So the training
# code gets its own interpreter, and the notebook kernel is left alone.
DS_ENV = CONTENT / "ds_env"          # python 3.11 + CUDA torch: everything but export
DS_ENV_PYTHON_VERSION = "3.11"
ONNX_ENV = CONTENT / "onnx_env"      # python 3.10 + CPU torch 1.13: ONNX export only
ONNX_ENV_PYTHON_VERSION = "3.10"

# Environment fixes applied to every subprocess we launch.
#   MPLBACKEND       Colab exports module://matplotlib_inline.backend_inline, which
#                    only exists in the kernel. Anything importing matplotlib in the
#                    venv (torchmetrics does) dies on "not a valid value for backend".
#   PYTHONPATH       Colab points this at /env/python, i.e. 3.13 site-packages. A
#                    3.11 interpreter must not see those.
#   PYTHONNOUSERSITE Same reasoning for ~/.local.
SUBPROCESS_ENV = {
    "MPLBACKEND": "Agg",
    "PYTHONPATH": "",
    "PYTHONNOUSERSITE": "1",
}

# --------------------------------------------------------------------------- #
#  Errors + pretty output
# --------------------------------------------------------------------------- #


class NotebookError(RuntimeError):
    """A problem we understand and can explain. Never a bare traceback."""


# Colab renders ANSI colour in cell output, which keeps the [ok]/[warn] lines readable
# at a glance. Log files get the same text; the escapes are harmless there.
_C = {"green": "\033[92m", "cyan": "\033[96m", "yellow": "\033[93m", "red": "\033[91m",
      "magenta": "\033[95m", "bold": "\033[1m", "dim": "\033[2m", "end": "\033[0m"}


def _html(markup):
    """Show rich HTML when running inside a notebook. Returns False elsewhere."""
    try:
        from IPython import get_ipython
        from IPython.display import HTML, display
        if get_ipython() is None:
            return False
        display(HTML(markup))
        return True
    except Exception:
        return False


def _esc(text):
    return str(text).replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")


_PANEL = ("font-family:Inter,Segoe UI,Roboto,sans-serif;border-radius:12px;"
          "padding:14px 18px;margin:12px 0 6px;color:#fff;")


def die(problem, fix=None, detail=None):
    """Print a readable failure panel and stop the cell."""
    width = 78
    print("\n" + "=" * width)
    print(f"  {_C['red']}{_C['bold']}STOPPED:{_C['end']} " + problem)
    if detail:
        print("-" * width)
        for line in str(detail).rstrip().splitlines()[-30:]:
            print("  " + line)
    if fix:
        print("-" * width)
        print("  How to fix:")
        for line in str(fix).strip().splitlines():
            print("    " + line)
    print("=" * width + "\n")
    _html(f'<div style="{_PANEL}background:linear-gradient(135deg,#dc2626,#7f1d1d);">'
          f'<div style="font-size:16px;font-weight:700">&#9940; Stopped: {_esc(problem)}</div>'
          + (f'<div style="margin-top:8px;font-size:13px"><b>How to fix</b><br>'
             f'<span style="white-space:pre-wrap;font-family:monospace">'
             f'{_esc(str(fix).strip())}</span></div>' if fix else "")
          + "</div>")
    # SystemExit rather than an ordinary exception: the panel above already says
    # everything useful, and IPython prints SystemExit without a traceback.
    raise SystemExit(problem)


def ok(msg):
    print(f"  {_C['green']}✔{_C['end']}  " + str(msg))


def info(msg):
    print(f"  {_C['cyan']}ℹ{_C['end']}  " + str(msg))


def warn(msg):
    print(f"  {_C['yellow']}⚠  " + str(msg) + _C["end"])


DEFAULT_ACCENT = ("#6366f1", "#8b5cf6")
FAST_ACCENT = ("#f97316", "#db2777")
DONE_ACCENT = ("#059669", "#0d9488")


def header(title, subtitle=None, accent=DEFAULT_ACCENT):
    """A gradient banner in Colab; a ruled text header anywhere else."""
    shown = _html(
        f'<div style="{_PANEL}background:linear-gradient(135deg,{accent[0]},{accent[1]});'
        f'box-shadow:0 4px 14px rgba(0,0,0,.18)">'
        f'<div style="font-size:17px;font-weight:700;letter-spacing:.2px">{_esc(title)}</div>'
        + (f'<div style="font-size:12.5px;opacity:.9;margin-top:3px">{_esc(subtitle)}</div>'
           if subtitle else "")
        + "</div>")
    if not shown:
        print("\n" + "=" * 78)
        print("  " + title + (f"  -  {subtitle}" if subtitle else ""))
        print("=" * 78)


def human_time(seconds):
    seconds = int(seconds)
    if seconds < 60:
        return f"{seconds}s"
    return f"{seconds // 60}m {seconds % 60:02d}s"


def human_size(num_bytes):
    value = float(num_bytes)
    for unit in ("B", "KB", "MB", "GB", "TB"):
        if value < 1024 or unit == "TB":
            return f"{value:.1f} {unit}"
        value /= 1024
    return f"{value:.1f} TB"


@contextlib.contextmanager
def step(name):
    """Time a stage and always say whether it worked."""
    print(f"\n{_C['bold']}{_C['magenta']}▶ {name}{_C['end']}")
    started = time.time()
    try:
        yield
    except NotebookError:
        raise
    except Exception as exc:
        print(f"{_C['red']}✖ {name} FAILED after {human_time(time.time() - started)}{_C['end']}")
        die(f"{name} failed", detail=f"{type(exc).__name__}: {exc}")
    else:
        print(f"{_C['dim']}  └ done in {human_time(time.time() - started)}{_C['end']}")


# --------------------------------------------------------------------------- #
#  Running commands
# --------------------------------------------------------------------------- #


def run(cmd, cwd=None, check=True, quiet=False, log_file=None, env=None):
    """
    Run a command from a *list* of arguments.

    Everything the original notebook did with `!cmd {python_var}` broke as soon
    as a path contained a space, and swallowed the exit code.  This does not:
    arguments are passed through without a shell, and a non-zero exit raises
    with the tail of the output attached.
    """
    if isinstance(cmd, str):
        cmd = shlex.split(cmd)
    cmd = [str(part) for part in cmd]

    LOG_DIR.mkdir(parents=True, exist_ok=True)
    handle = open(log_file, "a", encoding="utf-8", errors="replace") if log_file else None
    tail = []
    merged_env = {**os.environ, **SUBPROCESS_ENV, **(env or {})}

    process = subprocess.Popen(
        cmd,
        cwd=str(cwd) if cwd else None,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        errors="replace",
        env=merged_env,
    )
    try:
        for line in process.stdout:
            if handle:
                handle.write(line)
            tail.append(line)
            if len(tail) > 200:
                del tail[0]
            if not quiet:
                sys.stdout.write(line)
        process.wait()
    except KeyboardInterrupt:
        # Stop the child too. Without this, an interrupted training cell leaves a
        # python process holding the GPU and the next run dies with "out of memory".
        process.terminate()
        try:
            process.wait(timeout=10)
        except subprocess.TimeoutExpired:
            process.kill()
        print("\n  [info] interrupted - child process stopped")
        raise
    finally:
        if handle:
            handle.close()

    if check and process.returncode != 0:
        die(
            f"command failed (exit {process.returncode}): {' '.join(cmd[:4])} ...",
            fix=f"Full log: {log_file}" if log_file else None,
            detail="".join(tail),
        )
    return process.returncode


def run_out(cmd):
    """Run and return stdout as text; empty string if the command fails."""
    if isinstance(cmd, str):
        cmd = shlex.split(cmd)
    try:
        return subprocess.run(
            [str(c) for c in cmd], capture_output=True, text=True, errors="replace"
        ).stdout.strip()
    except Exception:
        return ""


def have(binary):
    return shutil.which(binary) is not None


# --------------------------------------------------------------------------- #
#  Interpreters
# --------------------------------------------------------------------------- #


def uv_path(log_file=None, quiet=True):
    """Path to `uv`, installing it into the kernel first if need be.

    uv ships as a self-contained binary, so this is a few seconds and cannot
    disturb the kernel's own packages.
    """
    found = shutil.which("uv")
    if found:
        return found
    run([sys.executable, "-m", "pip", "install", "-q", "uv"],
        log_file=log_file, quiet=quiet)
    found = shutil.which("uv")
    if found:
        return found
    candidate = Path(sys.executable).parent / "uv"
    if candidate.exists():
        return str(candidate)
    die("could not install uv",
        fix="Run `!pip install uv` in a new cell, then run setup again.")


def env_python(env_dir):
    candidate = Path(env_dir) / "bin" / "python"
    return str(candidate) if candidate.exists() else None


def training_python():
    """The interpreter that actually has DiffSinger's dependencies."""
    found = env_python(DS_ENV)
    if found is None:
        die("the DiffSinger environment is missing",
            fix="Run step 1 (Setup). It builds a Python "
                f"{DS_ENV_PYTHON_VERSION} environment at {DS_ENV}.")
    return found


def req_name(requirement_line):
    """'librosa<0.10.0' -> 'librosa'."""
    # maxsplit by keyword: passing it positionally is deprecated as of python 3.13.
    return re.split(r"[<>=!~\[; ]", str(requirement_line).strip(), maxsplit=1)[0].lower()


def venv_extras(requirements):
    """
    Packages every DiffSinger venv needs but no requirements file lists.

    A venv is not a Colab-wide install: things that are simply present system-wide
    have to be asked for.
    """
    present = {req_name(entry) for entry in requirements}
    extras = []
    if "setuptools" not in present:
        # lightning and librosa both `import pkg_resources`, which lives in
        # setuptools and is gone from 81 onwards.
        extras.append("setuptools<81")
    if "onnxsim" in present and "onnxruntime" not in present:
        # onnxsim shells out to `pip install onnxruntime` on first import if it is
        # missing, which fails inside a venv and takes the import down with it.
        extras.append("onnxruntime")
    return extras


def build_env(env_dir, python_version, requirements, log_file=None, quiet=True):
    """
    Create a throwaway interpreter and install `requirements` into it.

    Everything goes in ONE resolve, torch included.  Installing torch first and
    the rest afterwards does not work: `requirements-onnx.txt` leaves torchmetrics
    unpinned, modern torchmetrics wants torch>=2, and the second pass happily
    upgrades the torch you just pinned - which is how you end up with torch 2.14
    in an environment that asked for 1.13.1.  A single resolve makes the pin
    binding and forces the resolver to pick companions that fit it.
    """
    env_dir = Path(env_dir)
    requirements = list(requirements) + venv_extras(requirements)
    uv = uv_path(log_file=log_file, quiet=quiet)
    run([uv, "python", "install", python_version], log_file=log_file, quiet=quiet)
    # --seed puts pip and setuptools in the venv. Without it there is no
    # pkg_resources, and both lightning and librosa import it on sight.
    run([uv, "venv", "--seed", "--python", python_version, str(env_dir)],
        log_file=log_file, quiet=quiet)
    python = env_dir / "bin" / "python"

    spec = env_dir.with_suffix(".requirements.txt")
    spec.write_text("\n".join(requirements) + "\n", encoding="utf-8")
    run([uv, "pip", "install", "--python", str(python), "-r", str(spec)],
        log_file=log_file, quiet=quiet)
    return str(python)


def assert_pinned(python, module, expected, log_file=None):
    """
    Confirm a package really is the version we asked for.

    A resolver that moves torch is the single most destructive thing that can
    happen to one of these environments, and it fails much later and much less
    legibly than it should.
    """
    actual = run_out([str(python), "-c",
                      f"import {module};print({module}.__version__)"])
    if not actual:
        warn(f"could not read {module}.__version__ from {python}")
        return None
    if not actual.startswith(expected):
        die(f"{module} is {actual}, but this environment pinned {expected}",
            fix="Something in the requirements pulled a different version in.\n"
                "Re-run this cell with the force/rebuild option ticked. If it happens\n"
                "again, the log shows which package demanded it.\n"
                + (f"Log: {log_file}" if log_file else ""))
    return actual


def module_versions(python, checks):
    """[(module, pip_name)] -> rows of {module, version} using that interpreter.

    Results are collected first and printed last behind a sentinel, because some
    packages print during import - onnxsim shells out to pip - and anything they
    write to stdout would otherwise be parsed as if it were a result row.
    """
    probe = (
        "import importlib.metadata as md\n"
        "rows = []\n"
        "for module, dist in %r:\n"
        "    try:\n"
        "        __import__(module)\n"
        "    except Exception as exc:\n"
        "        reason = type(exc).__name__ + ': ' + str(exc).replace(chr(10), ' ')[:70]\n"
        "        rows.append((module, 'MISSING', reason))\n"
        "        continue\n"
        "    try:\n"
        "        rows.append((module, md.version(dist), ''))\n"
        "    except Exception:\n"
        "        rows.append((module, 'ok', ''))\n"
        "for row in rows:\n"
        "    print('@@ROW@@' + chr(9).join(row))\n"
    ) % (list(checks),)
    result = subprocess.run([str(python), "-c", probe], capture_output=True, text=True,
                            env={**os.environ, **SUBPROCESS_ENV})
    rows, broken = [], []
    for line in result.stdout.splitlines():
        if not line.startswith("@@ROW@@"):
            continue
        parts = line[len("@@ROW@@"):].split("\t")
        while len(parts) < 3:
            parts.append("")
        rows.append({"module": parts[0], "version": parts[1]})
        if parts[1] == "MISSING":
            broken.append((parts[0], parts[2] or "import failed"))
    if not rows:
        die("could not query the environment",
            detail=(result.stderr or result.stdout)[-1500:],
            fix="Re-run setup with force_reinstall ticked.")
    return rows, broken


# --------------------------------------------------------------------------- #
#  Downloading
# --------------------------------------------------------------------------- #


def download_many(items, log_file=None, quiet=True):
    """
    items: list of (url, destination_path).

    Uses aria2c with 16 connections per file and 4 files at a time, which is
    where most of the original setup's wall-clock went - it fetched ~1.5 GB of
    checkpoints one at a time with single-connection wget.
    """
    items = [(url, Path(dest)) for url, dest in items]
    todo = []
    for url, dest in items:
        if dest.exists() and dest.stat().st_size > 0:
            info(f"already have {dest.name} ({human_size(dest.stat().st_size)})")
        else:
            todo.append((url, dest))
    if not todo:
        return

    if have("aria2c"):
        spec = CONTENT / ".aria2_input"
        lines = []
        for url, dest in todo:
            dest.parent.mkdir(parents=True, exist_ok=True)
            lines.append(url)
            lines.append(f"  dir={dest.parent}")
            lines.append(f"  out={dest.name}")
        spec.write_text("\n".join(lines) + "\n", encoding="utf-8")
        run(
            [
                "aria2c", "--input-file", str(spec),
                "--max-concurrent-downloads", "4",
                "--max-connection-per-server", "16",
                "--split", "16",
                "--min-split-size", "1M",
                "--continue", "true",
                "--auto-file-renaming", "false",
                "--allow-overwrite", "true",
                "--console-log-level", "warn",
                "--summary-interval", "0",
            ],
            log_file=log_file,
            quiet=quiet,
        )
        spec.unlink(missing_ok=True)
    else:
        for url, dest in todo:
            dest.parent.mkdir(parents=True, exist_ok=True)
            run(["curl", "-L", "--fail", "--silent", "--show-error", "-o", str(dest), url],
                log_file=log_file, quiet=quiet)

    missing = [dest.name for _, dest in todo if not dest.exists() or dest.stat().st_size == 0]
    if missing:
        die("some downloads did not arrive: " + ", ".join(missing),
            fix="Re-run this cell. If it keeps failing the release URL may have moved.")
    for _, dest in todo:
        ok(f"{dest.name} ({human_size(dest.stat().st_size)})")


def unzip(archive, dest):
    """Extract a zip with the stdlib - no dependency on p7zip being present."""
    archive, dest = Path(archive), Path(dest)
    dest.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive) as zf:
        zf.extractall(dest)


def extract_archive(archive, dest):
    """
    Extract .zip/.7z/.rar/.tar.* for user datasets.

    Prefers 7z when available (handles rar/7z), falls back to Python for zip and
    tar so a missing p7zip is not a hard failure like it was before.
    """
    archive, dest = Path(archive), Path(dest)
    dest.mkdir(parents=True, exist_ok=True)
    suffix = archive.suffix.lower()

    if have("7z") and suffix in (".7z", ".rar", ".zip"):
        run(["7z", "x", "-y", str(archive), f"-o{dest}"], quiet=True)
        return
    if suffix == ".zip":
        unzip(archive, dest)
        return
    if suffix in (".gz", ".bz2", ".xz", ".tar", ".tgz"):
        import tarfile

        with tarfile.open(archive) as tf:
            tf.extractall(dest)
        return
    if have("7z"):
        run(["7z", "x", "-y", str(archive), f"-o{dest}"], quiet=True)
        return
    die(f"do not know how to extract {archive.name}",
        fix="Re-pack your dataset as a .zip file and try again.")


# --------------------------------------------------------------------------- #
#  Cross-cell state
# --------------------------------------------------------------------------- #


def load_state(required=None):
    if not STATE_PATH.exists():
        die("no saved notebook state found",
            fix="Run the Setup cell (step 1) first. It creates /content/ds_state.json.")
    state = json.loads(STATE_PATH.read_text(encoding="utf-8"))
    for key in required or []:
        if key not in state:
            die(f"state is missing '{key}'",
                fix=_state_hint(key))
    return state


def save_state(**values):
    state = {}
    if STATE_PATH.exists():
        try:
            state = json.loads(STATE_PATH.read_text(encoding="utf-8"))
        except Exception:
            state = {}
    state.update(values)
    STATE_PATH.write_text(json.dumps(state, indent=2), encoding="utf-8")
    return state


def _state_hint(key):
    hints = {
        "setup_done": "Run step 1 (Setup).",
        "data_dir": "Run step 2 (Prepare dataset).",
        "dictionary_files": "Run step 2 (Prepare dataset).",
        "config_path": "Run step 3 (Build config).",
        "model_dir": "Run step 3 (Build config).",
    }
    return hints.get(key, "Re-run the earlier cells in order.")


# --------------------------------------------------------------------------- #
#  Environment reporting
# --------------------------------------------------------------------------- #


def gpu_report():
    out = run_out(
        ["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
         "--format=csv,noheader"]
    )
    if not out:
        return None
    name, mem, driver = [part.strip() for part in out.splitlines()[0].split(",")]
    return {"name": name, "memory": mem, "driver": driver}


def gpu_specs():
    """Name, VRAM in GB and CUDA compute capability of the first GPU, or None."""
    out = run_out(["nvidia-smi", "--query-gpu=name,memory.total,compute_cap",
                   "--format=csv,noheader,nounits"])
    if not out:
        return None
    try:
        name, mem, cap = [part.strip() for part in out.splitlines()[0].split(",")]
        return {"name": name, "vram_gb": float(mem) / 1024, "compute_cap": float(cap)}
    except ValueError:
        return None


# --------------------------------------------------------------------------- #
#  Training modes
# --------------------------------------------------------------------------- #

ORIGINAL_MODE = "Original"
FAST_MODE = "Woah's Epic Faster Training (experimental)"


def is_fast(mode):
    return str(mode).startswith("Woah")


def fast_binarize_workers():
    """
    Parallel binarize workers. Each one loads its own copy of the pitch and
    harmonic/noise models, so leave a core for the kernel on big runtimes. On a
    2-core T4 two workers still win: while one waits on the GPU (RMVPE, VR) the
    other is decoding audio.
    """
    return min(8, max(2, cpu_count() - 1))


def fast_train_overrides(model_type, gpu=None, optimizer_cls=""):
    """
    What Woah's fast mode layers on top of the config: (dotted.key, value, reason).

    Every key is a stock DiffSinger hparam - nothing is patched, and Original mode
    stays exactly the stock behaviour. Where the speed comes from:
      - the right precision for the card: bf16 on Ampere+ (L4, A100), fp16 tensor
        cores on a T4 - which also rescues a T4 from bf16 / fp32 picked in step 3,
        both of which run at a fraction of fp16 speed there
      - fused AdamW: one CUDA kernel per step instead of one per tensor
      - batches sized to the GPU's VRAM instead of one T4-safe number
      - more dataloader workers and deeper prefetch so the GPU never waits on CPU
      - cheaper validation and logging
    torch.compile is handled separately by the launcher (FAST_LAUNCHER below).
    """
    gpu = gpu if gpu is not None else gpu_specs()
    vram = gpu["vram_gb"] if gpu else 15.0
    cap = gpu["compute_cap"] if gpu else 7.5

    # (min VRAM GB, max_batch_size, max_batch_frames). Variance models cost far
    # less per frame, which is why upstream already defaults them higher.
    if model_type == "acoustic":
        tiers = [(70, 64, 160000), (36, 48, 100000), (22, 32, 64000), (0, 24, 48000)]
    else:
        tiers = [(70, 96, 240000), (36, 64, 160000), (22, 48, 120000), (0, 32, 80000)]
    batch, frames = next((b, f) for limit, b, f in tiers if vram >= limit)

    # The training loop itself needs a core. Colab's T4 runtime has two, and two
    # loader workers there starve the process that launches the GPU kernels;
    # DiffSinger's loading is light (pre-binarized arrays), so one keeps up.
    workers = max(1, min(8, cpu_count() - 1))
    rows = [
        ("max_batch_size", batch, f"sized for {vram:.0f} GB of VRAM"),
        ("max_batch_frames", frames, "fills the GPU instead of the T4-safe default"),
        ("ds_workers", workers, f"{cpu_count()} CPU cores: one left for the training loop"),
        ("dataloader_prefetch_factor", 4, "keeps batches queued ahead of the GPU"),
        ("num_valid_plots", 3, "validation renders 3 samples instead of 10"),
        ("num_sanity_val_steps", 0, "skips the warm-up validation pass"),
        ("log_interval", 500, "TensorBoard scalars every 500 steps, not 100"),
    ]
    name = gpu["name"] if gpu else "this GPU"
    if cap >= 8.0:
        rows.insert(0, ("pl_trainer_precision", "bf16-mixed",
                        f"{name} supports bfloat16 natively"))
    else:
        rows.insert(0, ("pl_trainer_precision", "16-mixed",
                        f"fp16 tensor cores - bf16 is emulated on {name}"))
    # Under fp16, Lightning refuses gradient clipping (DiffSinger always clips) on
    # an optimizer that unscales gradients itself, which fused AdamW does. Muon_AdamW
    # wraps its AdamW in a ChainedOptimizer that leaves unscaling to Lightning, so
    # fused is safe there at any precision; plain AdamW only gets it under bf16.
    if optimizer_cls.endswith("Muon_AdamW"):
        rows.append(("optimizer_args.adamw_args.fused", True, "fused AdamW kernel"))
    elif cap >= 8.0:
        rows.append(("optimizer_args.fused", True, "fused AdamW kernel (bf16)"))
    return rows


# Keys fast mode may touch. Original mode puts each back to its stock value, so
# flipping the dropdown back after a fast run really is the stock setup again.
FAST_KEYS = ("pl_trainer_precision", "max_batch_size", "max_batch_frames", "ds_workers",
             "dataloader_prefetch_factor", "num_valid_plots", "num_sanity_val_steps",
             "log_interval", "optimizer_args.fused", "optimizer_args.adamw_args.fused")


def _dotted(cfg, key, value=None, delete=False):
    *parents, leaf = key.split(".")
    node = cfg
    for part in parents:
        if not isinstance(node.get(part), dict):
            if delete:
                return
            node[part] = {}
        node = node[part]
    if delete:
        node.pop(leaf, None)
    else:
        node[leaf] = value


def apply_training_mode(cfg, rows=None, stock=None):
    """Fast mode: write `rows`. Original mode: restore FAST_KEYS from `stock`."""
    if rows is not None:
        for key, value, _ in rows:
            _dotted(cfg, key, value)
        return
    stock = stock or {}
    for key in FAST_KEYS:
        value = stock
        for part in key.split("."):
            value = value.get(part) if isinstance(value, dict) else None
        if value is None:
            _dotted(cfg, key, delete=True)
        else:
            _dotted(cfg, key, value)


FAST_ENV = {
    # TF32 for whatever still runs in fp32 on Ampere+. No effect on a T4.
    "TORCH_ALLOW_TF32_CUBLAS_OVERRIDE": "1",
    # Fewer allocator stalls and less fragmentation with variable-length batches.
    "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True",
    # Read by FAST_LAUNCHER: redraw the progress bar every 10 steps, not every step.
    "NB_PROGRESS_EVERY": "10",
}

# Fast-mode launcher. It applies runtime tweaks that have no config key, then hands
# over to the unmodified scripts/train.py. The DiffSinger tree is never edited.
#   - TF32 matmuls / convolutions for anything still in fp32 (Ampere+)
#   - progress bar redrawn every NB_PROGRESS_EVERY steps instead of every step.
#     Each redraw formats every metric and pushes a line through the pipe to the
#     notebook; on a 2-core T4 that is CPU time stolen from launching kernels.
#   - torch.compile on the diffusion backbone, which is where nearly all of the
#     FLOPs are. The class `forward` is wrapped, not the module, so state_dict
#     keys - and therefore checkpoints and ONNX export - are unchanged.
#     dynamic=True because DiffSinger batches vary in length every step.
FAST_LAUNCHER = r'''
import os
import runpy
import sys

repo = sys.argv[1]
script = os.path.join(repo, "scripts", "train.py")
sys.argv = [script] + sys.argv[2:]
sys.path.insert(0, repo)

import torch

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

every = int(os.environ.get("NB_PROGRESS_EVERY", "1"))
if every > 1:
    try:
        import utils.training_utils as training_utils
        bar_init = training_utils.DsTQDMProgressBar.__init__

        def init_with_refresh(self, refresh_rate=1, *args, **kwargs):
            bar_init(self, every, *args, **kwargs)

        training_utils.DsTQDMProgressBar.__init__ = init_with_refresh
    except Exception as exc:
        print(f"| nb: progress bar left as-is ({type(exc).__name__}: {exc})", flush=True)

if os.environ.get("NB_TORCH_COMPILE") == "1":
    try:
        import torch._dynamo
        # A graph dynamo cannot handle runs eagerly instead of killing training.
        torch._dynamo.config.suppress_errors = True
        torch._dynamo.config.cache_size_limit = 64
        import modules.backbones as backbones
        for name, cls in backbones.BACKBONES.items():
            cls.forward = torch.compile(cls.forward, dynamic=True)
        print("| nb: torch.compile enabled for backbones: " + ", ".join(backbones.BACKBONES),
              flush=True)
    except Exception as exc:
        print(f"| nb: torch.compile unavailable ({type(exc).__name__}: {exc}) - running eager",
              flush=True)

runpy.run_path(script, run_name="__main__")
'''

# Log lines that mean the compiler, not the model, killed a run.
COMPILE_FAILURE_MARKERS = ("torch._dynamo", "torch._inductor", "BackendCompilerFailed",
                           "InductorError", "triton.compiler", "CompilationError")


def write_fast_launcher():
    path = CONTENT / "nb_fast_train.py"
    path.write_text(FAST_LAUNCHER.lstrip("\n"), encoding="utf-8")
    return path


def disk_free(path="/content"):
    usage = shutil.disk_usage(path)
    return usage.free


def ram_total():
    try:
        for line in Path("/proc/meminfo").read_text().splitlines():
            if line.startswith("MemTotal:"):
                return int(line.split()[1]) * 1024
    except Exception:
        pass
    return 0


def cpu_count():
    return os.cpu_count() or 2


# --------------------------------------------------------------------------- #
#  Checkpoints / experiment plumbing
# --------------------------------------------------------------------------- #


SAFE_NAME = re.compile(r"[^A-Za-z0-9._-]+")


def sanitize_exp_name(name):
    cleaned = SAFE_NAME.sub("_", str(name).strip()).strip("._-")
    return cleaned or "model"


def link_experiment(exp_name, target_dir):
    """
    Make `DiffSinger/checkpoints/<exp_name>` point at the user's model folder.

    DiffSinger computes its work dir as `checkpoints/<exp_name>`, so a symlink
    is all that is needed to train straight into Google Drive.  The original
    notebook instead rewrote utils/hparams.py and utils/training_utils.py with
    regexes on every run, which broke resuming and broke checkpoint deletion.
    """
    target_dir = Path(target_dir)
    target_dir.mkdir(parents=True, exist_ok=True)
    link = REPO / "checkpoints" / exp_name
    link.parent.mkdir(parents=True, exist_ok=True)
    if link.is_symlink():
        link.unlink()
    elif link.is_dir():
        # A real directory of the same name - never delete it silently.
        if link.resolve() != target_dir.resolve():
            die(f"checkpoints/{exp_name} already exists as a real folder",
                fix=f"Rename your model folder, or remove {link}")
        return link
    elif link.exists():
        link.unlink()
    link.symlink_to(target_dir, target_is_directory=True)
    return link


CKPT_STEPS = re.compile(r"steps_(\d+)")


def find_latest_ckpt(folder):
    folder = Path(folder)
    best, best_step = None, -1
    for path in folder.glob("*.ckpt"):
        match = CKPT_STEPS.search(path.name)
        step_num = int(match.group(1)) if match else 0
        if step_num > best_step:
            best, best_step = path, step_num
    return best


# --------------------------------------------------------------------------- #
#  Config loading / validation
# --------------------------------------------------------------------------- #


def load_yaml(path):
    import yaml

    with open(path, "r", encoding="utf-8") as fh:
        return yaml.safe_load(fh) or {}


def dump_yaml(data, path):
    import yaml

    with open(path, "w", encoding="utf-8") as fh:
        yaml.dump(data, fh, sort_keys=False, allow_unicode=True)


def resolve_merged_config(config_path):
    """
    Reproduce DiffSinger's base_config chaining so we can validate the *effective*
    config, not just the leaf file.
    """
    config_path = Path(config_path)
    seen = set()

    def load_chain(path):
        path = Path(path).resolve()
        if path in seen:
            return {}
        seen.add(path)
        raw = load_yaml(path)
        merged = {}
        bases = raw.get("base_config") or []
        if isinstance(bases, str):
            bases = [bases]
        for base in bases:
            base_path = base
            if str(base).startswith("."):
                base_path = os.path.normpath(os.path.join(path.parent, base))
            elif not os.path.isabs(base):
                base_path = os.path.normpath(os.path.join(REPO, base))
            merged.update(load_chain(base_path))
        merged.update({k: v for k, v in raw.items() if k != "base_config"})
        return merged

    return load_chain(config_path)


def preflight(config_path, model_type):
    """
    Check everything binarize/train will reach for *before* spending 20 minutes
    finding out the hard way.  Returns nothing; raises with a specific fix.
    """
    cfg = resolve_merged_config(config_path)
    problems = []

    def check_file(key, label):
        value = cfg.get(key)
        if not value:
            return
        path = Path(value)
        if not path.is_absolute():
            path = REPO / path
        if not path.exists():
            problems.append(f"{label}: config '{key}' points at {value} which does not exist")

    if cfg.get("pe") == "rmvpe":
        check_file("pe_ckpt", "pitch extractor")
    wants_hnsep = any(
        cfg.get(k) for k in (
            "use_tension_embed", "use_voicing_embed", "use_breathiness_embed",
            "predict_tension", "predict_voicing", "predict_breathiness",
        )
    )
    if wants_hnsep and cfg.get("hnsep") == "vr":
        check_file("hnsep_ckpt", "harmonic/noise separator")
    if model_type == "acoustic":
        check_file("vocoder_ckpt", "vocoder")

    dictionaries = cfg.get("dictionaries") or {}
    for lang, dict_path in dictionaries.items():
        path = Path(dict_path)
        if not path.is_absolute():
            path = REPO / path
        if not path.exists():
            problems.append(f"dictionary '{lang}' missing: {dict_path}")

    languages_used = {ds.get("language") for ds in cfg.get("datasets") or []}
    unknown = {lang for lang in languages_used if lang and lang not in dictionaries}
    if unknown:
        problems.append(
            "dataset language(s) "
            + ", ".join(sorted(unknown))
            + " have no matching dictionary (have: "
            + ", ".join(sorted(dictionaries)) + ")"
        )

    for dataset in cfg.get("datasets") or []:
        raw_dir = Path(dataset.get("raw_data_dir", ""))
        if not raw_dir.exists():
            problems.append(f"raw_data_dir missing: {raw_dir}")
            continue
        if not (raw_dir / "transcriptions.csv").exists() and not list(raw_dir.glob("*.ds")):
            problems.append(f"{raw_dir.name}: no transcriptions.csv and no .ds files")

    binary_dir = cfg.get("binary_data_dir")
    if binary_dir:
        parent = Path(binary_dir).parent
        if not parent.exists():
            problems.append(f"binary_data_dir parent does not exist: {parent}")

    if problems:
        die("the config will not work as-is",
            fix="Fix the items above, then re-run the config cell.",
            detail="\n".join("- " + p for p in problems))
    ok("pre-flight checks passed")


# --------------------------------------------------------------------------- #
#  Dataset inspection
# --------------------------------------------------------------------------- #


def wav_seconds(path):
    """Duration from the WAV header only - no numpy, no librosa, instant."""
    try:
        with contextlib.closing(wave.open(str(path), "rb")) as wf:
            return wf.getnframes() / float(wf.getframerate())
    except Exception:
        return 0.0


def dataset_summary(data_dir):
    """Return per-speaker stats for a prepared DiffSinger dataset folder."""
    data_dir = Path(data_dir)
    rows = []
    for speaker_dir in sorted(p for p in data_dir.iterdir() if p.is_dir()):
        wavs = list((speaker_dir / "wavs").glob("*.wav"))
        ds_files = list(speaker_dir.glob("*.ds"))
        seconds = sum(wav_seconds(w) for w in wavs)
        rows.append({
            "speaker": speaker_dir.name,
            "segments": len(wavs) or len(ds_files),
            "hours": seconds / 3600.0,
            "has_csv": (speaker_dir / "transcriptions.csv").exists(),
        })
    return rows


def print_table(rows, columns):
    """Summary table: styled HTML in Colab, fixed-width text anywhere else."""
    if not rows:
        return
    cell = "padding:5px 14px;border-bottom:1px solid rgba(127,127,127,.25);text-align:left"
    markup = ('<table style="border-collapse:collapse;font-family:Inter,Segoe UI,Roboto,'
              'sans-serif;font-size:13px;margin:6px 0 10px">'
              + "<tr>" + "".join(f'<th style="{cell};font-weight:700;opacity:.75">{_esc(t)}</th>'
                                 for _, t in columns) + "</tr>"
              + "".join("<tr>" + "".join(f'<td style="{cell}">{_esc(row.get(k, ""))}</td>'
                                         for k, _ in columns) + "</tr>" for row in rows)
              + "</table>")
    if _html(markup):
        return
    widths = {}
    for key, title in columns:
        widths[key] = max(len(title), *(len(str(row.get(key, ""))) for row in rows))
    line = "  " + "  ".join(title.ljust(widths[key]) for key, title in columns)
    print(line)
    print("  " + "  ".join("-" * widths[key] for key, _ in columns))
    for row in rows:
        print("  " + "  ".join(str(row.get(key, "")).ljust(widths[key]) for key, _ in columns))

In [ ]:
#@title 1. Setup  -  installs everything (safe to re-run)
#@markdown Installs everything DiffSinger needs. Safe to re-run: finished steps are skipped.
#@markdown <br><font size="-1">Typical cold run: **4-7 minutes**. Re-runs: a few seconds.</font>

#@markdown ---
branch = "main"  #@param ["main", "muon_lynxnet2"]
#@markdown <font size="-1">`muon_lynxnet2` is experimental - things can and do break there.</font>

profile = "training"  #@param ["training", "onnx export only", "everything"]
#@markdown <font size="-1">`onnx export only` skips the ~1 GB of training checkpoints.</font>

vocoder_choice = "pc_nsf_hifigan (recommended)"  #@param ["pc_nsf_hifigan (recommended)", "nsf_hifigan", "both"]

mount_drive = True  #@param {type:"boolean"}
install_some = True  #@param {type:"boolean"}
#@markdown <font size="-1">SOME is only needed if you want it to estimate MIDI for variance/pitch training.</font>

show_full_log = False  #@param {type:"boolean"}
force_reinstall = False  #@param {type:"boolean"}

# --------------------------------------------------------------------------- #

import hashlib
import importlib
import shutil
import sys
import time
from pathlib import Path

sys.path.insert(0, "/content")
if not Path("/content/dsnb.py").exists():
    raise SystemExit("Run the cell above (step 0) first - it writes the helper library.")
import dsnb
importlib.reload(dsnb)
from dsnb import (CONTENT, DS_ENV_PYTHON_VERSION, LOG_DIR, REPO, die, download_many, have,
                  header, info, human_size, human_time, ok, print_table, run, run_out,
                  save_state, step, unzip, warn)

QUIET = not show_full_log
LOG_DIR.mkdir(parents=True, exist_ok=True)
LOG = LOG_DIR / "setup.log"
STAMP = CONTENT / ".setup_complete"
STARTED = time.time()

WANT_TRAINING = profile in ("training", "everything")
WANT_EXPORT = profile in ("onnx export only", "everything")

REPOS = {
    "DiffSinger": ("https://github.com/usamireko/DiffSinger.git", branch),
    "nnsvs-db-converter": ("https://github.com/UtaUtaUtau/nnsvs-db-converter", None),
    "MakeDiffSinger": ("https://github.com/openvpi/MakeDiffSinger", None),
    "ghin_shenanigans": ("https://github.com/MLo7Ghinsan/ghin_shenanigans", None),
    "SOME": ("https://github.com/openvpi/SOME", None),
}

ASSETS = {
    "rmvpe": (
        "https://github.com/openvpi/DiffSinger/releases/download/v2.1.0/rmvpe.zip",
        CONTENT / "dl" / "rmvpe.zip",
    ),
    "hnsep": (
        "https://github.com/yxlllc/vocal-remover/releases/download/hnsep_240512/hnsep_240512.zip",
        CONTENT / "dl" / "hnsep_240512.zip",
    ),
    "pc_nsf_hifigan": (
        "https://github.com/openvpi/vocoders/releases/download/"
        "pc-nsf-hifigan-44.1k-hop512-128bin-2025.02/"
        "pc_nsf_hifigan_44.1k_hop512_128bin_2025.02.zip",
        CONTENT / "dl" / "pc_nsf_hifigan.zip",
    ),
    "nsf_hifigan": (
        "https://github.com/openvpi/vocoders/releases/download/"
        "nsf-hifigan-44.1k-hop512-128bin-2024.02/"
        "nsf_hifigan_44.1k_hop512_128bin_2024.02.zip",
        CONTENT / "dl" / "nsf_hifigan.zip",
    ),
    "some": (
        "https://github.com/openvpi/SOME/releases/download/v1.0.0-baseline/"
        "0119_continuous128_5spk.zip",
        CONTENT / "dl" / "some_baseline.zip",
    ),
}

TARGET_PYTHON = tuple(int(part) for part in DS_ENV_PYTHON_VERSION.split("."))

# torch is pinned here rather than taken from requirements.txt (which does not
# list it at all - it says "install PyTorch manually").
#   2.5.1 because lightning~=2.3 predates torch 2.6, where torch.load flipped to
#   weights_only=True by default and checkpoint loading breaks.
TORCH_PIN = "torch==2.5.1"
TORCHAUDIO_PIN = "torchaudio==2.5.1"

# Packages we deliberately do NOT install, and why.
#   torch / torchvision / torchaudio  -> pinned above instead, so the resolver
#                                        cannot wander onto an incompatible one.
#   SOME/requirements.txt             -> pins fairseq==0.12.2 + gradio==3.47.1 and
#                                        onnx==1.14.0, which fight DiffSinger's
#                                        onnx~=1.16.0 and take ~30 min to resolve.
#                                        SOME's batch_infer only imports click,
#                                        librosa, tqdm and yaml, all of which
#                                        DiffSinger already requires.
SKIP_PACKAGES = {"torch", "torchvision", "torchaudio"}

# Not in requirements.txt but used by the MakeDiffSinger scripts. (setuptools and
# onnxruntime are added by dsnb.venv_extras, which every venv gets.)
EXTRA_PACKAGES = ["mido"]

# Pins with no wheel for a given Python or newer. Empty in practice at 3.11 -
# which is the whole point of not using the kernel's 3.13.
PIN_OVERRIDES = {
    "praat-parselmouth": ("praat-parselmouth>=0.4.5", (3, 12)),
}

IMPORT_CHECKS = [
    ("torch", "torch"),
    ("lightning", "lightning"),
    ("numpy", "numpy"),
    ("librosa", "librosa"),
    ("parselmouth", "praat-parselmouth"),
    ("pyworld", "pyworld-prebuilt"),
    ("onnx", "onnx"),
    ("onnxruntime", "onnxruntime"),
    ("onnxsim", "onnxsim"),
    ("yaml", "PyYAML"),
    ("einops", "einops"),
    ("tensorboard", "tensorboard"),
    ("torchmetrics", "torchmetrics"),
    ("matplotlib", "matplotlib"),
    ("h5py", "h5py"),
    ("resampy", "resampy"),
    ("mido", "mido"),
    ("pkg_resources", "setuptools"),
]


# --------------------------------------------------------------------------- #
header("DiffSinger setup")

if STAMP.exists() and not force_reinstall:
    info("setup already completed in this runtime - verifying instead of reinstalling")
    info("tick 'force_reinstall' if you want it done again from scratch")

# --- environment ----------------------------------------------------------- #
gpu = dsnb.gpu_report()
free = dsnb.disk_free("/content")
print(f"  python      {sys.version.split()[0]}")
print(f"  gpu         {gpu['name'] + ' (' + gpu['memory'] + ')' if gpu else 'NONE - training will not work'}")
print(f"  cpu / ram   {dsnb.cpu_count()} cores / {human_size(dsnb.ram_total())}")
print(f"  disk free   {human_size(free)}")

if not gpu and WANT_TRAINING:
    warn("no GPU detected. Runtime > Change runtime type > T4 GPU, then re-run.")
if free < 12 * 1024 ** 3 and WANT_TRAINING:
    warn(f"only {human_size(free)} free - binarized data plus checkpoints usually needs 15 GB+")

# --- drive ----------------------------------------------------------------- #
if mount_drive:
    with step("Mounting Google Drive"):
        if Path("/content/drive/MyDrive").exists():
            ok("already mounted")
        else:
            from google.colab import drive

            drive.mount("/content/drive")
            ok("mounted at /content/drive")

# --- system packages ------------------------------------------------------- #
with step("System packages"):
    needed = [pkg for pkg, binary in (("aria2", "aria2c"), ("p7zip-full", "7z"),
                                      ("unzip", "unzip")) if not have(binary)]
    if needed:
        if run(["apt-get", "install", "-y", "-qq"] + needed,
               log_file=LOG, quiet=QUIET, check=False) != 0:
            # Stale package lists are the usual cause and the usual cure.
            info("retrying after apt-get update")
            run(["apt-get", "update", "-qq"], log_file=LOG, quiet=QUIET)
            run(["apt-get", "install", "-y", "-qq"] + needed, log_file=LOG, quiet=QUIET)
        ok("installed " + ", ".join(needed))
    else:
        ok("aria2 / 7z / unzip already present")

# --- repositories ---------------------------------------------------------- #
with step("Repositories"):
    for name, (url, want_branch) in REPOS.items():
        if name == "SOME" and not install_some:
            continue
        target = CONTENT / name
        if (target / ".git").exists():
            current = run_out(["git", "-C", str(target), "rev-parse", "--abbrev-ref", "HEAD"])
            if want_branch and current != want_branch:
                info(f"{name}: switching {current} -> {want_branch}")
                run(["git", "-C", str(target), "remote", "set-branches", "origin", want_branch],
                    log_file=LOG, quiet=QUIET)
                run(["git", "-C", str(target), "fetch", "--depth", "1", "origin", want_branch],
                    log_file=LOG, quiet=QUIET)
                run(["git", "-C", str(target), "checkout", "-B", want_branch,
                     f"origin/{want_branch}"], log_file=LOG, quiet=QUIET)
            ok(f"{name} present ({want_branch or current})")
            continue
        cmd = ["git", "clone", "--depth", "1", "--quiet"]
        if want_branch:
            cmd += ["-b", want_branch]
        cmd += [url, str(target)]
        run(cmd, log_file=LOG, quiet=QUIET)
        ok(f"{name} cloned" + (f" ({want_branch})" if want_branch else ""))

if not (REPO / "requirements.txt").exists():
    die("DiffSinger did not clone correctly",
        fix="Re-run this cell. If it still fails, Runtime > Disconnect and delete runtime.")

# --- python environment ---------------------------------------------------- #
if not WANT_TRAINING:
    DS_PYTHON = None
    info("export-only profile: step 6 builds its own python 3.10 environment")
else:
    with step(f"DiffSinger environment (python {DS_ENV_PYTHON_VERSION})"):
        requirements, dropped, retargeted = [], [], []
        for raw in (REPO / "requirements.txt").read_text(encoding="utf-8").splitlines():
            entry = raw.split("#", 1)[0].strip()
            if not entry:
                continue
            name = dsnb.req_name(entry)
            if name in SKIP_PACKAGES:
                dropped.append(entry)
                continue
            override, min_python = PIN_OVERRIDES.get(name, (None, None))
            if override and TARGET_PYTHON >= min_python:
                retargeted.append(f"{entry} -> {override}")
                entry = override
            requirements.append(entry)
        requirements += EXTRA_PACKAGES

        fingerprint = hashlib.sha256(
            ("|".join([DS_ENV_PYTHON_VERSION, TORCH_PIN] + requirements)).encode()
        ).hexdigest()[:16]
        marker = CONTENT / ".ds_env_complete"
        marker_matches = (marker.exists()
                          and marker.read_text(encoding="utf-8").strip() == fingerprint)

        print(f"  kernel python   {sys.version.split()[0]}  (left untouched)")
        print(f"  target python   {DS_ENV_PYTHON_VERSION} at {dsnb.DS_ENV}")
        print(f"  torch           {TORCH_PIN}")
        for line in retargeted:
            info(line)
        if dropped:
            info("torch is pinned separately, not taken from requirements.txt: "
                 + ", ".join(dropped))
        print(f"  packages        {len(requirements)}: " + ", ".join(requirements))

        existing = dsnb.env_python(dsnb.DS_ENV)
        broken = dsnb.module_versions(existing, IMPORT_CHECKS)[1] if existing else []
        reusable = bool(existing) and marker_matches and not broken and not force_reinstall

        if reusable:
            ok("already built and everything imports - nothing to do")
            DS_PYTHON = existing
        else:
            if existing and marker_matches and broken:
                warn("the environment exists but these do not import: "
                     + ", ".join(module for module, _ in broken))
                warn("rebuilding")
            marker.unlink(missing_ok=True)
            if dsnb.DS_ENV.exists():
                shutil.rmtree(dsnb.DS_ENV)

            print("  ---- install output ------------------------------------------------")
            # Deliberately NOT quiet. This is the longest step, so watching it is the
            # difference between "working" and "hung".
            # torch goes in the same resolve as everything else - see build_env.
            DS_PYTHON = dsnb.build_env(
                dsnb.DS_ENV,
                DS_ENV_PYTHON_VERSION,
                [TORCH_PIN, TORCHAUDIO_PIN] + requirements,
                log_file=LOG,
                quiet=False,
            )
            dsnb.assert_pinned(DS_PYTHON, "torch",
                               TORCH_PIN.split("==")[1], log_file=LOG)
            print("  ---- end install output --------------------------------------------")
            marker.write_text(fingerprint, encoding="utf-8")
            ok(f"built at {dsnb.DS_ENV}")

# --- checkpoints ----------------------------------------------------------- #
checkpoints = REPO / "checkpoints"
checkpoints.mkdir(parents=True, exist_ok=True)

if WANT_TRAINING:
    wanted = ["rmvpe", "hnsep"]
    if vocoder_choice != "nsf_hifigan":
        wanted.append("pc_nsf_hifigan")
    if vocoder_choice in ("nsf_hifigan", "both"):
        wanted.append("nsf_hifigan")
    if install_some:
        wanted.append("some")

    with step(f"Downloading {len(wanted)} checkpoint archives in parallel"):
        download_many([ASSETS[key] for key in wanted], log_file=LOG, quiet=QUIET)

    with step("Installing checkpoints"):
        # rmvpe -> checkpoints/rmvpe/model.pt  (and a copy MakeDiffSinger expects)
        if not (checkpoints / "rmvpe" / "model.pt").exists():
            unzip(ASSETS["rmvpe"][1], checkpoints)
        assets_dir = CONTENT / "MakeDiffSinger" / "variance-temp-solution" / "assets"
        if assets_dir.exists() and not (assets_dir / "rmvpe" / "model.pt").exists():
            unzip(ASSETS["rmvpe"][1], assets_dir)
        ok("rmvpe -> checkpoints/rmvpe/model.pt")

        # hnsep: every config ships `hnsep_ckpt: checkpoints/vr/model.pt`, but the
        # release unpacks to hnsep_240512/.  The original notebook left it there,
        # so tension / voicing / breathiness extraction could never find it.
        vr_dir = checkpoints / "vr"
        if not (vr_dir / "model.pt").exists():
            staging = CONTENT / ".hnsep_tmp"
            if staging.exists():
                shutil.rmtree(staging)
            unzip(ASSETS["hnsep"][1], staging)
            found = next(staging.rglob("model.pt"), None)
            if found is None:
                die("hnsep archive did not contain model.pt",
                    fix="Delete /content/dl/hnsep_240512.zip and re-run this cell.")
            vr_dir.mkdir(parents=True, exist_ok=True)
            for item in found.parent.iterdir():
                shutil.move(str(item), vr_dir / item.name)
            shutil.rmtree(staging, ignore_errors=True)
        ok("hnsep -> checkpoints/vr/model.pt  (matches hnsep_ckpt in the configs)")

        for key in ("pc_nsf_hifigan", "nsf_hifigan"):
            if key not in wanted:
                continue
            prefix = "pc_nsf_hifigan_44.1k" if key.startswith("pc_") else "nsf_hifigan_44.1k"
            found = [p for p in checkpoints.glob(prefix + "*")
                     if p.is_dir() and (p / "model.ckpt").exists()]
            if not found:
                unzip(ASSETS[key][1], checkpoints)
                found = [p for p in checkpoints.glob(prefix + "*") if p.is_dir()]
            if found:
                ok(f"{key} -> checkpoints/{found[0].name}")
            else:
                warn(f"{key} did not unpack as expected")

        some_ckpt = None
        if install_some:
            some_root = checkpoints / "SOME"
            if not list(some_root.rglob("*.ckpt")):
                unzip(ASSETS["some"][1], some_root)
            # Do not hard-code the folder name.  The original notebook looked for
            # `0119_continuous256_5spk` while downloading `...128_5spk`, so SOME
            # MIDI estimation could never start.
            candidates = sorted(some_root.rglob("*.ckpt"))
            simplified = [c for c in candidates if "simplified" in c.name]
            some_ckpt = str((simplified or candidates)[0]) if candidates else None
            if some_ckpt:
                ok(f"SOME -> {Path(some_ckpt).relative_to(CONTENT)}")
            else:
                warn("SOME checkpoint not found; MIDI estimation via SOME will be unavailable")
else:
    some_ckpt = None
    info("export-only profile: skipping training checkpoints")

# --- verify ---------------------------------------------------------------- #
if not DS_PYTHON:
    info("export-only profile: nothing to verify here")
else:
    with step("Verifying the environment"):
        # A version for every module. "Did it actually install?" should never be
        # something you have to guess at.
        rows, broken = dsnb.module_versions(DS_PYTHON, IMPORT_CHECKS)
        print_table(rows, [("module", "module"), ("version", "version")])

        if broken:
            print()
            for module, reason in broken:
                warn(f"{module}: {reason}")
            by_module = dict(IMPORT_CHECKS)
            die(f"{len(broken)} module(s) failed to import",
                fix="Install them into the DiffSinger environment:\n"
                    f"  !{dsnb.uv_path()} pip install --python {DS_PYTHON} "
                    + " ".join(by_module[m] for m, _ in broken)
                    + f"\nThen re-run this cell. Full log: {LOG}")
        ok(f"all {len(rows)} required modules import cleanly")

        cuda = run_out([DS_PYTHON, "-c",
                        "import torch;print(torch.__version__, torch.cuda.is_available())"])
        if cuda:
            version, available = cuda.split()
            if available == "True":
                ok(f"torch {version} sees the GPU")
            elif WANT_TRAINING:
                warn(f"torch {version} but CUDA is NOT available - training will be unusably slow")
            warn("could not query torch in the DiffSinger environment")

# --------------------------------------------------------------------------- #
save_state(
    setup_done=True,
    branch=branch,
    profile=profile,
    repo=str(REPO),
    some_ckpt=some_ckpt,
    vocoder_choice=vocoder_choice,
    ds_python=DS_PYTHON,
)
STAMP.write_text("ok", encoding="utf-8")

header(f"Setup finished in {human_time(time.time() - STARTED)}", accent=dsnb.DONE_ACCENT)
if DS_PYTHON:
    print(f"  DiffSinger runs on {DS_PYTHON}")
    print(f"  This notebook's kernel stays on python {sys.version.split()[0]} and is untouched.")
print("  Next: step 2 - prepare your dataset.")
print(f"  Log:  {LOG}")

---
## &#128193; Dataset and config

Step 2 unpacks and segments your data. Step 3 turns your choices into a config, picks your
**training mode**, and checks it will actually work. Step 4 binarizes.

Your archive should contain **one folder per singer**:

```
data.zip
 +- my_singer/
     +- song1.wav
     +- song1.lab
     +- ...
```

For multiple languages, add a `lang_config.yaml` at the top level and suffix each
folder with its language code (`my_singer.ja`, `other_singer.en`).

In [ ]:
#@title 2. Prepare dataset  -  unpack, segment, write dictionaries
#@markdown Unpacks your data into `/content/raw_data`, segments it, and writes the phoneme
#@markdown dictionaries. Prints a summary so you can see what actually made it through.

#@markdown ---
#@markdown ### Input
data_type = "lab + wav (NNSVS)"  #@param ["lab + wav (NNSVS)", "csv + wav (DiffSinger)", "ds (DiffSinger)"]
data_zip_path = ""  #@param {type:"string"}
#@markdown <font size="-1">Path to your archive on Drive - `.zip`, `.7z` and `.rar` all work, spaces included.</font>

#@markdown ---
#@markdown ### Segmentation &nbsp;<font size="-1">(lab + wav only)</font>
segment_length = 10  #@param {type:"slider", min:5, max:35, step:1}
max_silences = 2  #@param {type:"slider", min:0, max:20, step:1}
detect_breaths = True  #@param {type:"boolean"}
#@markdown <font size="-1">`detect_breaths` finds AP inside pauses automatically - recommended.</font>

normalize_silence_labels = True  #@param {type:"boolean"}
#@markdown <font size="-1">Rewrites `SP`/`sil` to `pau` and `br` to `AP` in your labels so the segmenter understands them.</font>

write_ds_and_labels = False  #@param {type:"boolean"}
#@markdown <font size="-1">Also emit `.ds` files and HTK labels beside the segments - handy for
#@markdown checking segmentation in SlurCutter, and for the "package dataset" utility at the bottom.</font>

#@markdown ---
#@markdown ### MIDI estimation &nbsp;<font size="-1">(needed for variance *pitch* training)</font>
estimate_midi = "off"  #@param ["off", "parselmouth", "harvest", "SOME (most accurate)"]

# --------------------------------------------------------------------------- #

import csv
import importlib
import json
import shutil
import sys
from pathlib import Path

sys.path.insert(0, "/content")
import dsnb
importlib.reload(dsnb)
from dsnb import (CONTENT, REPO, dataset_summary, die, extract_archive, header, info,
                  load_state, ok, print_table, run, save_state, step, warn)

state = load_state(required=["setup_done"])
PY = dsnb.training_python()   # DiffSinger's python 3.11 env, not the notebook kernel

RAW_ROOT = CONTENT / "raw_data"
DATA_DIR = RAW_ROOT / "diffsinger_db"
DICT_DIR = REPO / "dictionaries"
LANG_JSON = CONTENT / "nnsvs-db-converter" / "lang.sample.json"

SILENCE_TOKENS = {"pau", "AP", "SP", "sil", "br", "silB", "silE", ""}
LABEL_FIXES = {"SP": "pau", "sil": "pau", "silB": "pau", "silE": "pau", "br": "AP"}
VOWEL_STARTS = {"a", "i", "u", "e", "o", "A", "I", "U", "E", "O", "N", "M"}
LIQUID_STARTS = {"y", "w", "l", "r"}

header("Prepare dataset")

if not data_zip_path.strip():
    die("no data_zip_path given",
        fix="Paste the full path to your dataset archive, e.g.\n"
            "  /content/drive/MyDrive/my singer/data.zip")
archive = Path(data_zip_path.strip().strip('"').strip("'"))
if not archive.exists():
    die(f"archive not found: {archive}",
        fix="Check the path. Use the file browser on the left, right-click > Copy path.")


# --------------------------------------------------------------------------- #
#  1. Extract
# --------------------------------------------------------------------------- #
with step(f"Extracting {archive.name}"):
    if RAW_ROOT.exists():
        shutil.rmtree(RAW_ROOT)
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    extract_archive(archive, DATA_DIR)

    # Tolerate an archive that wraps everything in one extra folder.
    entries = [p for p in DATA_DIR.iterdir() if not p.name.startswith(".")]
    dirs = [p for p in entries if p.is_dir()]
    if len(dirs) == 1 and len(entries) == 1:
        inner = dirs[0]
        looks_like_speaker = any(inner.glob("*.lab")) or any(inner.glob("*.wav")) \
            or (inner / "wavs").exists() or any(inner.glob("*.ds"))
        if not looks_like_speaker:
            info(f"unwrapping extra top-level folder '{inner.name}'")
            for item in list(inner.iterdir()):
                shutil.move(str(item), DATA_DIR / item.name)
            inner.rmdir()

    speaker_dirs = sorted(p for p in DATA_DIR.iterdir() if p.is_dir())
    if not speaker_dirs:
        die("no speaker folders found inside the archive",
            fix="Your archive must contain one folder per singer, e.g.\n"
                "  data.zip\n"
                "   +- my_singer/\n"
                "       +- *.wav and *.lab")
    ok(f"{len(speaker_dirs)} speaker folder(s): " + ", ".join(p.name for p in speaker_dirs))


# --------------------------------------------------------------------------- #
#  2. Normalise label files
#
#  The original notebook ran this on the csv/ds branch instead of the lab branch
#  ("using 'if not' bc i edited the wrong section"), so it never touched a single
#  .lab file.  It also did a raw string replace, which corrupted any phoneme that
#  merely contained "br" or "SP".  This rewrites the phoneme column only.
# --------------------------------------------------------------------------- #
if data_type == "lab + wav (NNSVS)" and normalize_silence_labels:
    with step("Normalising label files"):
        changes, touched = {}, 0
        for lab in DATA_DIR.rglob("*.lab"):
            lines, dirty = [], False
            for line in lab.read_text(encoding="utf-8", errors="replace").splitlines():
                parts = line.split()
                if len(parts) >= 3 and parts[2] in LABEL_FIXES:
                    changes[parts[2]] = changes.get(parts[2], 0) + 1
                    parts[2] = LABEL_FIXES[parts[2]]
                    dirty = True
                    lines.append(" ".join(parts))
                else:
                    lines.append(line)
            if dirty:
                lab.write_text("\n".join(lines) + "\n", encoding="utf-8")
                touched += 1
        if changes:
            ok(f"{touched} file(s) updated: "
               + ", ".join(f"{k}->{LABEL_FIXES[k]} x{v}" for k, v in sorted(changes.items())))
        else:
            ok("labels already use pau / AP - nothing to change")


# --------------------------------------------------------------------------- #
#  3. Collect phonemes and write dictionaries
# --------------------------------------------------------------------------- #
def phonemes_in(path):
    """Yield phonemes from a .lab, .csv or .ds file, minus silence tokens."""
    suffix = path.suffix.lower()
    try:
        if suffix == ".lab":
            for line in path.read_text(encoding="utf-8", errors="replace").splitlines():
                parts = line.split()
                if len(parts) >= 3:
                    yield parts[2]
        elif suffix == ".csv":
            with path.open("r", newline="", encoding="utf-8", errors="replace") as fh:
                for row in csv.DictReader(fh):
                    for token in (row.get("ph_seq") or "").split():
                        yield token
        elif suffix == ".ds":
            payload = json.loads(path.read_text(encoding="utf-8", errors="replace"))
            for entry in payload if isinstance(payload, list) else [payload]:
                for token in (entry.get("ph_seq") or "").split():
                    yield token
    except Exception as exc:
        warn(f"could not read {path.name}: {exc}")


def strip_lang(token):
    return token.split("/", 1)[1] if "/" in token else token


lang_config_path = DATA_DIR / "lang_config.yaml"
extra_phonemes, merged_phoneme_groups = [], []
speakers = []          # [{folder, speaker, language}]
lang_phonemes = {}     # language -> set(phonemes)

with step("Building phoneme dictionaries"):
    if lang_config_path.exists():
        lang_config = dsnb.load_yaml(lang_config_path)
        languages = list(lang_config.get("languages") or [])
        extra_phonemes = list(lang_config.get("extra_phonemes") or [])
        merged_phoneme_groups = list(lang_config.get("merged_phoneme_groups") or [])
        if not languages:
            die("lang_config.yaml has no 'languages' list",
                fix="Add e.g.\n  languages:\n    - ja\n    - en")
        info("multi-language mode: " + ", ".join(languages))
        lang_phonemes = {lang: set() for lang in languages}

        unlabelled = []
        for folder in speaker_dirs:
            if "." not in folder.name:
                unlabelled.append(folder.name)
                continue
            base, _, suffix = folder.name.rpartition(".")
            if suffix not in languages:
                unlabelled.append(folder.name)
                continue
            speakers.append({"folder": str(folder), "speaker": base, "language": suffix})
            for path in folder.rglob("*"):
                if path.suffix.lower() not in (".lab", ".csv", ".ds"):
                    continue
                for token in phonemes_in(path):
                    hint, _, tail = token.partition("/")
                    if tail and hint in languages:
                        if tail not in SILENCE_TOKENS:
                            lang_phonemes[hint].add(tail)
                    elif token not in SILENCE_TOKENS:
                        lang_phonemes[suffix].add(token)

        if unlabelled:
            die("these folders have no language suffix: " + ", ".join(unlabelled),
                fix="With lang_config.yaml present, every speaker folder must end in a\n"
                    "language code from the list, e.g. 'my_singer.ja'.")
    else:
        info("single-language mode (no lang_config.yaml) - dictionary will be 'custom'")
        collected = set()
        for folder in speaker_dirs:
            speakers.append({"folder": str(folder), "speaker": folder.name, "language": "custom"})
            for path in folder.rglob("*"):
                if path.suffix.lower() not in (".lab", ".csv", ".ds"):
                    continue
                for token in phonemes_in(path):
                    token = strip_lang(token)
                    if token not in SILENCE_TOKENS:
                        collected.add(token)
        lang_phonemes = {"custom": collected}

    if not any(lang_phonemes.values()):
        die("no phonemes found anywhere in the dataset",
            fix={"lab + wav (NNSVS)": "Expected *.lab files next to your *.wav files.",
                 "csv + wav (DiffSinger)": "Expected transcriptions.csv with a ph_seq column.",
                 "ds (DiffSinger)": "Expected *.ds files with a ph_seq field."}[data_type])

    DICT_DIR.mkdir(parents=True, exist_ok=True)
    dictionaries, dictionary_files = {}, []
    for lang, phoneme_set in lang_phonemes.items():
        path = DICT_DIR / f"dictionary-{lang}.txt"
        path.write_text(
            "".join(f"{ph}\t{ph}\n" for ph in sorted(phoneme_set)), encoding="utf-8"
        )
        dictionaries[lang] = str(path)
        dictionary_files.append(str(path))
        ok(f"{path.name}: {len(phoneme_set)} phonemes")

    # Language definition for the segmenter's MIDI estimator.
    all_phonemes = sorted({ph for group in lang_phonemes.values() for ph in group})
    vowels = [p for p in all_phonemes if p[0] in VOWEL_STARTS]
    liquids = {p: True for p in all_phonemes if p[0] in LIQUID_STARTS}
    LANG_JSON.parent.mkdir(parents=True, exist_ok=True)
    LANG_JSON.write_text(json.dumps({"vowels": vowels, "liquids": liquids}, indent=2),
                         encoding="utf-8")
    info(f"{len(vowels)} vowels / {len(liquids)} liquids written for the segmenter")


# --------------------------------------------------------------------------- #
#  4. Convert to DiffSinger layout
# --------------------------------------------------------------------------- #
if data_type == "lab + wav (NNSVS)":
    converter = CONTENT / "nnsvs-db-converter" / "db_converter.py"
    use_some = estimate_midi.startswith("SOME")
    inline_midi = estimate_midi in ("parselmouth", "harvest")

    with step("Segmenting audio"):
        for entry in speakers:
            folder = Path(entry["folder"])
            cmd = [PY, str(converter),
                   "-s", str(max_silences),
                   "-l", str(segment_length),
                   "-T", "0",                      # use every core; the original used one
                   "-L", str(LANG_JSON)]
            if detect_breaths:
                cmd.append("-B")
            if write_ds_and_labels:
                cmd += ["-D", "-w", "htk"]
            if inline_midi:
                # The original hard-coded parselmouth: it read the menu choice into
                # `midi_pitch_ext` and then never passed it to anything.
                cmd += ["-m", "-c", "-p", estimate_midi]
            cmd.append(str(folder))
            info(f"{folder.name}: segmenting (this is the slow part)")
            run(cmd, quiet=True)

            produced = folder / "diffsinger_db"
            if not produced.exists():
                die(f"the segmenter produced nothing for '{folder.name}'",
                    fix="Check that every .wav has a matching .lab with the same name.")
            for stale in list(folder.glob("*.wav")) + list(folder.glob("*.lab")):
                stale.unlink()
            for item in produced.iterdir():
                destination = folder / item.name
                if destination.exists():
                    shutil.rmtree(destination) if destination.is_dir() else destination.unlink()
                shutil.move(str(item), destination)
            produced.rmdir()
            segments = len(list((folder / "wavs").glob("*.wav")))
            ok(f"{folder.name}: {segments} segments")

    if use_some:
        some_ckpt = state.get("some_ckpt")
        if not some_ckpt or not Path(some_ckpt).exists():
            die("SOME checkpoint is not available",
                fix="Re-run step 1 with 'install_some' ticked.")
        with step("Estimating MIDI with SOME"):
            for entry in speakers:
                info(f"{Path(entry['folder']).name}: running SOME")
                run([PY, "batch_infer.py",
                     "--model", some_ckpt,
                     "--dataset", entry["folder"],
                     "--overwrite"],
                    cwd=CONTENT / "SOME", quiet=True)
                ok(f"{Path(entry['folder']).name}: note_seq written")

elif data_type == "ds (DiffSinger)":
    segmentor = CONTENT / "ghin_shenanigans" / "scripts" / "ds_segmentor.py"
    ds2csv = CONTENT / "MakeDiffSinger" / "variance-temp-solution" / "convert_ds.py"
    with step("Segmenting .ds and converting to transcriptions.csv"):
        for entry in speakers:
            folder = Path(entry["folder"])
            ds_out = folder / "ds"
            run([PY, str(segmentor), str(folder), "--export_path", str(ds_out)],
                quiet=True)
            for stale in folder.glob("*.ds"):
                stale.unlink()
            run([PY, str(ds2csv), "ds2csv", str(ds_out),
                 str(folder / "transcriptions.csv")], quiet=True)
            ok(f"{folder.name}: {len(list(ds_out.glob('*.ds')))} .ds files")
else:
    info("csv + wav data used as-is")


# --------------------------------------------------------------------------- #
#  5. Leading SP -> AP
#
#  A segment that opens on silence should open on AP, not SP.  The original only
#  ever patched row 1 of each file, leaving every other segment wrong.
# --------------------------------------------------------------------------- #
with step("Fixing leading silence tokens"):
    fixed_rows = 0
    for csv_path in DATA_DIR.rglob("transcriptions.csv"):
        with csv_path.open("r", newline="", encoding="utf-8") as fh:
            rows = list(csv.reader(fh))
        if not rows or "ph_seq" not in rows[0]:
            continue
        column = rows[0].index("ph_seq")
        changed = False
        for row in rows[1:]:
            if len(row) <= column:
                continue
            tokens = row[column].split()
            if tokens and tokens[0] == "SP":
                tokens[0] = "AP"
                row[column] = " ".join(tokens)
                fixed_rows += 1
                changed = True
        if changed:
            with csv_path.open("w", newline="", encoding="utf-8") as fh:
                csv.writer(fh).writerows(rows)
    ok(f"{fixed_rows} segment(s) now start on AP")


# --------------------------------------------------------------------------- #
#  6. Summary
# --------------------------------------------------------------------------- #
rows = dataset_summary(DATA_DIR)
for row in rows:
    row["hours"] = f"{row['hours']:.2f}"
    row["has_csv"] = "yes" if row["has_csv"] else "NO"

header("Dataset ready", accent=dsnb.DONE_ACCENT)
print_table(rows, [("speaker", "speaker"), ("segments", "segments"),
                   ("hours", "hours"), ("has_csv", "transcriptions")])

missing_csv = [r["speaker"] for r in rows if r["has_csv"] == "NO"]
if missing_csv:
    warn("no transcriptions.csv for: " + ", ".join(missing_csv))

total_hours = sum(float(r["hours"]) for r in rows)
print()
info(f"{len(rows)} speaker(s), {sum(int(r['segments']) for r in rows)} segments, "
     f"{total_hours:.2f} hours total")
if total_hours and total_hours < 0.5:
    warn("under 30 minutes of audio - expect a rough model")

save_state(
    data_dir=str(DATA_DIR),
    data_type=data_type,
    speakers=speakers,
    dictionaries=dictionaries,
    dictionary_files=dictionary_files,
    extra_phonemes=extra_phonemes,
    merged_phoneme_groups=merged_phoneme_groups,
    languages=sorted(lang_phonemes),
    has_midi=estimate_midi != "off",
    total_hours=total_hours,
)
print("\n  Next: step 3 - build the training config.")

In [ ]:
#@title 3. Build training config  -  and check it will actually work
#@markdown Writes `configs/<your model>.yaml` inside the DiffSinger repo. Your stock
#@markdown configs are never modified, so this cell is safe to re-run as often as you like.

#@markdown ---
#@markdown ### Training mode
training_mode = "Original"  #@param ["Original", "Woah's Epic Faster Training (experimental)"]
#@markdown <font size="-1">**Original** is the stock DiffSinger setup, exactly as before.<br>
#@markdown **Woah's Epic Faster Training** &#9889; binarizes on every CPU core, forces local-disk
#@markdown binary data, and in step 5 compiles the backbone, picks the fastest precision for your
#@markdown GPU (fp16 on a T4, bf16 on L4/A100), sizes batches to it and cuts CPU and validation
#@markdown overhead. Tuned for the free T4 first. Only stock
#@markdown DiffSinger settings are changed. *Experimental:* if you hit CUDA out-of-memory, switch
#@markdown back to Original.</font>

#@markdown ---
#@markdown ### Model
model_type = "acoustic"  #@param ["acoustic", "variance"]
model_dir = "/content/drive/MyDrive/DiffSinger/my_singer_acoustic"  #@param {type:"string"}
#@markdown <font size="-1">Folder for this model's checkpoints - **edit the example above**. It is
#@markdown created for you. Keep it under `/content/drive/MyDrive/` or you lose it when Colab
#@markdown disconnects. Use a **separate folder per model**, since a full voicebank needs an
#@markdown acoustic *and* a variance model. This is also what you paste into step 5 to resume
#@markdown and into step 6 to export.</font>

diffusion_type = "reflow"  #@param ["reflow", "ddpm"]
backbone = "lynxnet"  #@param ["lynxnet", "lynxnet2", "wavenet"]
#@markdown <font size="-1">`lynxnet2` requires the `muon_lynxnet2` branch in step 1.</font>
sampling_algorithm = "euler"  #@param ["euler", "rk2", "rk4", "rk5"]
diff_accelerator = "unipc"  #@param ["unipc", "ddim", "pndm", "dpm-solver"]
loss_type = "l2"  #@param ["l2", "l1"]
precision = "16-mixed"  #@param ["16-mixed", "bf16-mixed", "32-true"]

#@markdown ---
#@markdown ### Features to train
#@markdown <font size="-1">Acoustic *embeds* and variance *predictions* must match, or OpenUtau
#@markdown will ask for a parameter your model cannot produce. This cell checks that for you.</font>
tension = True  #@param {type:"boolean"}
voicing = True  #@param {type:"boolean"}
breathiness = False  #@param {type:"boolean"}
energy = False  #@param {type:"boolean"}
#@markdown <font size="-1">Energy is deprecated upstream; leave it off unless you know why you want it.</font>

#@markdown <font size="-1">Variance only:</font>
duration_training = True  #@param {type:"boolean"}
pitch_training = "off"  #@param ["off", "standard", "melody encoder"]

#@markdown ---
#@markdown ### Extraction
f0_extractor = "rmvpe"  #@param ["rmvpe", "parselmouth", "harvest"]
parameter_extraction = "vr"  #@param ["vr", "world"]
shallow_diffusion = "on (gt_val)"  #@param ["on (gt_val)", "on (aux_val)", "off"]

#@markdown ---
#@markdown ### Fine-tuning
finetune = False  #@param {type:"boolean"}
base_model_path = ""  #@param {type:"string"}
#@markdown <font size="-1">Leave blank to download haru0l's public base model for the selected model type.</font>

#@markdown ---
#@markdown ### Storage
binary_location = "local disk (fastest)"  #@param ["local disk (fastest)", "next to the model (Drive)"]
#@markdown <font size="-1">Reading binarized data over Drive's FUSE mount slows training down noticeably.
#@markdown Local disk is rebuilt by re-running step 4 if the runtime dies.</font>

#@markdown ---
#@markdown ### Network size &nbsp;<font size="-1">- leave these alone unless you know what they do</font>
acoustic_hidden_size = 256  #@param {type:"slider", min:64, max:1024, step:32}
acoustic_num_layers = 6  #@param {type:"slider", min:2, max:24, step:1}
acoustic_num_channels = 1024  #@param {type:"slider", min:64, max:2048, step:64}
variance_hidden_size = 256  #@param {type:"slider", min:64, max:1024, step:32}
duration_hidden_size = 512  #@param {type:"slider", min:64, max:1024, step:32}
melody_encoder_hidden_size = 128  #@param {type:"slider", min:32, max:512, step:32}
pitch_num_layers = 6  #@param {type:"slider", min:2, max:24, step:1}
pitch_num_channels = 512  #@param {type:"slider", min:64, max:1024, step:64}
variance_num_layers = 6  #@param {type:"slider", min:2, max:24, step:1}
variance_num_channels = 384  #@param {type:"slider", min:64, max:1024, step:64}

# --------------------------------------------------------------------------- #

import csv
import importlib
import shutil
import sys
from pathlib import Path

sys.path.insert(0, "/content")
import dsnb
importlib.reload(dsnb)
from dsnb import (CONTENT, DEFAULT_ACCENT, FAST_ACCENT, REPO, die, download_many, dump_yaml,
                  header, info, is_fast, link_experiment, load_state, ok, preflight,
                  print_table, sanitize_exp_name, save_state, step, warn)

state = load_state(required=["setup_done", "data_dir", "dictionaries"])
DATA_DIR = Path(state["data_dir"])
speakers = state["speakers"]

FAST = is_fast(training_mode)
header(f"Build {model_type} config",
       subtitle=("⚡ Woah's Epic Faster Training mode (experimental)" if FAST
                 else "Original training mode"),
       accent=FAST_ACCENT if FAST else DEFAULT_ACCENT)

if not model_dir.strip():
    die("model_dir is empty",
        fix="Pick a folder for your checkpoints, e.g.\n"
            "  /content/drive/MyDrive/DiffSinger/my_singer_acoustic")
MODEL_DIR = Path(model_dir.strip().strip('"').strip("'"))
MODEL_DIR.mkdir(parents=True, exist_ok=True)
EXP_NAME = sanitize_exp_name(MODEL_DIR.name)

if not str(MODEL_DIR).startswith("/content/drive/"):
    warn(f"{MODEL_DIR} is not on Google Drive.")
    warn("Everything there is deleted when this Colab runtime ends - including")
    warn("your checkpoints. Use a path under /content/drive/MyDrive/ instead.")
elif not Path("/content/drive/MyDrive").exists():
    die("that path is on Drive, but Drive is not mounted",
        fix="Re-run step 1 with 'mount_drive' ticked.")

existing_ckpt = dsnb.find_latest_ckpt(MODEL_DIR)
if existing_ckpt:
    warn(f"{MODEL_DIR.name} already contains {existing_ckpt.name}.")
    warn("Training will continue from it rather than start over. For a fresh model,")
    warn("point model_dir at a new folder.")

if backbone == "lynxnet2" and state.get("branch") != "muon_lynxnet2":
    die("lynxnet2 needs the muon_lynxnet2 branch",
        fix="Re-run step 1 with branch = muon_lynxnet2, or pick a different backbone.")


with step("Copying dictionaries next to the model"):
    # Keeping the dictionaries inside the model folder makes it self-contained:
    # the config still resolves after a fresh runtime, and the OpenUtau builder
    # in step 7 can find them without you hunting for /content paths.
    dict_out = MODEL_DIR / "dictionaries"
    dict_out.mkdir(parents=True, exist_ok=True)
    dictionaries = {}
    for lang, source in state["dictionaries"].items():
        target = dict_out / Path(source).name
        shutil.copy(source, target)
        dictionaries[lang] = str(target)
        ok(f"{lang} -> {target.name}")


# --------------------------------------------------------------------------- #
#  Datasets block
# --------------------------------------------------------------------------- #
with step("Describing datasets"):
    datasets = []
    per_speaker_tests = max(1, 4 // max(1, len(speakers)))

    for spk_id, entry in enumerate(speakers):
        folder = Path(entry["folder"])
        if state["data_type"] == "ds (DiffSinger)":
            items = sorted(p.stem for p in folder.glob("*.ds"))
        else:
            items = sorted(p.stem for p in (folder / "wavs").glob("*.wav"))
        if not items:
            die(f"'{folder.name}' has no audio segments",
                fix="Re-run step 2; the segmenter did not produce anything for this speaker.")
        # Whole item names, not prefixes: a short prefix can silently match every
        # item in the set and leave the training split empty.
        chosen = items[:per_speaker_tests]
        datasets.append({
            "raw_data_dir": str(folder),
            "speaker": entry["speaker"],
            "spk_id": spk_id,
            "language": entry["language"],
            "test_prefixes": chosen,
        })
        ok(f"spk_id {spk_id}: {entry['speaker']} [{entry['language']}] - {len(items)} items")

    num_spk = len(datasets)
    languages = sorted({d["language"] for d in datasets})
    # use_lang_id depends on how many languages are in play. The original tied it
    # to merged_phoneme_groups, which is a different setting entirely.
    use_lang_id = len(dictionaries) > 1
    if len(dictionaries) > 1 and len(languages) == 1:
        info("multiple dictionaries but one language in use - keeping use_lang_id on")


# --------------------------------------------------------------------------- #
#  Fine-tuning
# --------------------------------------------------------------------------- #
PRETRAINED = {
    "acoustic": "https://github.com/haru0l/diffsinger_models/releases/download/"
                "acoustic/model_ckpt_steps_49000.ckpt",
    "variance": "https://github.com/haru0l/diffsinger_models/releases/download/"
                "variance/model_ckpt_steps_51000.ckpt",
}

finetune_ckpt = None
if finetune:
    with step("Preparing base model"):
        if base_model_path.strip():
            finetune_ckpt = Path(base_model_path.strip().strip('"').strip("'"))
            if not finetune_ckpt.exists():
                die(f"base model not found: {finetune_ckpt}",
                    fix="Check the path, or clear it to use the public base model.")
        else:
            finetune_ckpt = CONTENT / "pretrain_models" / f"{model_type}_pretrain.ckpt"
            download_many([(PRETRAINED[model_type], finetune_ckpt)], quiet=True)
        # The original set finetune_ckpt_path only on the *download* branch, so a
        # custom base_model_path raised NameError a few lines later.
        ok(f"fine-tuning from {finetune_ckpt}")


# --------------------------------------------------------------------------- #
#  Storage
# --------------------------------------------------------------------------- #
if FAST and not binary_location.startswith("local"):
    info("fast mode keeps binarized data on local disk - Drive's FUSE mount is the slowest")
    info("part of a training step. Tick 'backup_binary_to_drive' in step 4 for a safety copy.")
    binary_location = "local disk (fastest)"

if binary_location.startswith("local"):
    BINARY_DIR = CONTENT / "binary" / EXP_NAME
else:
    BINARY_DIR = MODEL_DIR / "binary"
BINARY_DIR.parent.mkdir(parents=True, exist_ok=True)

with step("Linking experiment folder"):
    link = link_experiment(EXP_NAME, MODEL_DIR)
    ok(f"checkpoints/{EXP_NAME} -> {MODEL_DIR}")
    info("DiffSinger resolves its work dir as checkpoints/<exp_name>, so this symlink is")
    info("all it takes to train straight into Drive - no source files are patched.")


# --------------------------------------------------------------------------- #
#  Compose the override config
# --------------------------------------------------------------------------- #
shallow_on = shallow_diffusion.startswith("on")
gt_start = shallow_diffusion == "on (gt_val)"
predict_pitch = pitch_training != "off"

cfg = {
    "base_config": [str(REPO / "configs" / f"{model_type}.yaml")],
    "datasets": datasets,
    "num_spk": num_spk,
    "use_spk_id": num_spk > 1,
    "dictionaries": dictionaries,
    "num_lang": len(dictionaries),
    "use_lang_id": use_lang_id,
    "extra_phonemes": state.get("extra_phonemes") or [],
    "merged_phoneme_groups": state.get("merged_phoneme_groups") or [],
    "binary_data_dir": str(BINARY_DIR),
    "pl_trainer_precision": precision,
    "diffusion_type": diffusion_type,
    "diff_accelerator": diff_accelerator,
    "main_loss_type": loss_type,
    "pe": f0_extractor,
    "pe_ckpt": "checkpoints/rmvpe/model.pt" if f0_extractor == "rmvpe" else None,
    "hnsep": parameter_extraction,
    "hnsep_ckpt": "checkpoints/vr/model.pt",
    "tension_smooth_width": 0.06,
    "finetune_enabled": bool(finetune),
    "finetune_ckpt_path": str(finetune_ckpt) if finetune_ckpt else None,
    "finetune_strict_shapes": not finetune,
    "pretrain": bool(finetune),
}

if model_type == "acoustic":
    # Point at whichever vocoder step 1 actually downloaded, so choosing the plain
    # nsf_hifigan there does not leave the config asking for the pc_ one.
    vocoder_dir = ("nsf_hifigan_44.1k_hop512_128bin_2024.02"
                   if state.get("vocoder_choice") == "nsf_hifigan"
                   else "pc_nsf_hifigan_44.1k_hop512_128bin_2025.02")
    cfg.update({
        "vocoder_ckpt": f"checkpoints/{vocoder_dir}/model.ckpt",
        "use_tension_embed": tension,
        "use_voicing_embed": voicing,
        "use_breathiness_embed": breathiness,
        "use_energy_embed": energy,
        "use_shallow_diffusion": shallow_on,
        "shallow_diffusion_args": {"val_gt_start": gt_start},
        "sampling_algorithm": sampling_algorithm,
        "hidden_size": acoustic_hidden_size,
        "backbone_type": backbone,
        "backbone_args": {"num_layers": acoustic_num_layers,
                          "num_channels": acoustic_num_channels},
        # Augmentation is on by default and stays on - it is close to free quality.
        "use_key_shift_embed": True,
        "use_speed_embed": True,
        "augmentation_args": {
            "random_pitch_shifting": {"enabled": True},
            "random_time_stretching": {"enabled": True},
        },
    })
else:
    cfg.update({
        "predict_tension": tension,
        "predict_voicing": voicing,
        "predict_breathiness": breathiness,
        "predict_energy": energy,
        "predict_dur": duration_training,
        "predict_pitch": predict_pitch,
        "use_melody_encoder": pitch_training == "melody encoder",
        "use_glide_embed": False,
        "hidden_size": variance_hidden_size,
        "dur_prediction_args": {"hidden_size": duration_hidden_size},
        "melody_encoder_args": {"hidden_size": melody_encoder_hidden_size},
        "variances_prediction_args": {
            "backbone_type": backbone,
            "backbone_args": {"num_layers": variance_num_layers,
                              "num_channels": variance_num_channels},
        },
        "pitch_prediction_args": {
            "backbone_type": backbone,
            "backbone_args": {"num_layers": pitch_num_layers,
                              "num_channels": pitch_num_channels},
        },
        "binarization_args": {"prefer_ds": state["data_type"] == "ds (DiffSinger)"},
    })

if FAST:
    # Parallel binarize. Stock is 0 (single process), which on a large dataset is
    # most of the time spent in step 4.
    cfg.setdefault("binarization_args", {})["num_workers"] = dsnb.fast_binarize_workers()

cfg = {key: value for key, value in cfg.items() if value is not None}


# --------------------------------------------------------------------------- #
#  Deep-merge our overrides onto the stock config so nested dicts keep their
#  untouched keys (setting backbone_args wholesale would drop kernel_size etc.)
# --------------------------------------------------------------------------- #
def deep_merge(base, override):
    merged = dict(base)
    for key, value in override.items():
        if isinstance(value, dict) and isinstance(merged.get(key), dict):
            merged[key] = deep_merge(merged[key], value)
        else:
            merged[key] = value
    return merged


with step("Writing config"):
    stock = dsnb.resolve_merged_config(REPO / "configs" / f"{model_type}.yaml")
    unknown = [k for k in cfg if k not in stock and k != "base_config"]
    resolved = {"base_config": cfg["base_config"]}
    for key, value in cfg.items():
        if key == "base_config":
            continue
        resolved[key] = deep_merge(stock[key], value) if (
            isinstance(value, dict) and isinstance(stock.get(key), dict)) else value

    CONFIG_PATH = REPO / "configs" / f"nb_{EXP_NAME}_{model_type}.yaml"
    dump_yaml(resolved, CONFIG_PATH)
    dump_yaml(resolved, MODEL_DIR / "notebook_config.yaml")   # copy for resuming later
    ok(f"{CONFIG_PATH}")
    if unknown:
        # These still take effect - hparams merges our file over the stock one - but
        # a key DiffSinger never reads is inert, so it is worth naming them.
        info("settings added on top of the stock config: " + ", ".join(sorted(unknown)))


# --------------------------------------------------------------------------- #
#  Sanity checks that save you an hour
# --------------------------------------------------------------------------- #
with step("Checking the setup makes sense"):
    if model_type == "variance" and predict_pitch:
        sample = next(DATA_DIR.rglob("transcriptions.csv"), None)
        if sample is None:
            die("pitch training needs transcriptions.csv", fix="Re-run step 2.")
        with sample.open("r", newline="", encoding="utf-8") as fh:
            columns = next(csv.reader(fh), [])
        if "note_seq" not in columns:
            die("pitch training is on but your data has no note_seq (MIDI)",
                fix="Re-run step 2 with 'estimate_midi' set to parselmouth, harvest or SOME.")
        ok("note_seq present - pitch training can proceed")

    if model_type == "variance" and not (duration_training or predict_pitch
                                         or tension or voicing or breathiness or energy):
        die("a variance model that predicts nothing has nothing to train",
            fix="Turn on duration, pitch, or at least one parameter.")

    preflight(CONFIG_PATH, model_type)

    if model_type == "variance":
        info("Reminder: your acoustic model must embed exactly the parameters this")
        info("variance model predicts (tension/voicing/breathiness/energy).")


# --------------------------------------------------------------------------- #
header("Config summary", accent=FAST_ACCENT if FAST else DEFAULT_ACCENT)
summary = [
    {"k": "training mode", "v": ("⚡ " + training_mode) if FAST else training_mode},
    {"k": "model", "v": f"{model_type} ({'multi' if num_spk > 1 else 'single'}-speaker)"},
    {"k": "speakers", "v": ", ".join(d["speaker"] for d in datasets)},
    {"k": "languages", "v": ", ".join(sorted(dictionaries))},
    {"k": "backbone", "v": f"{backbone} / {diffusion_type} / {diff_accelerator}"},
    {"k": "precision", "v": precision},
    {"k": "f0 / params", "v": f"{f0_extractor} / {parameter_extraction}"},
    {"k": "fine-tune", "v": str(finetune_ckpt) if finetune_ckpt else "no (training from scratch)"},
    {"k": "binary data", "v": str(BINARY_DIR)},
    {"k": "binarize workers", "v": str(dsnb.fast_binarize_workers()) if FAST else "0 (stock)"},
    {"k": "checkpoints", "v": str(MODEL_DIR)},
    {"k": "exp_name", "v": EXP_NAME},
    {"k": "config", "v": str(CONFIG_PATH)},
]
if model_type == "acoustic":
    summary.insert(3, {"k": "embeds", "v": ", ".join(
        [n for n, on in (("tension", tension), ("voicing", voicing),
                         ("breathiness", breathiness), ("energy", energy)) if on]) or "none"})
else:
    summary.insert(3, {"k": "predicts", "v": ", ".join(
        [n for n, on in (("duration", duration_training), ("pitch", predict_pitch),
                         ("tension", tension), ("voicing", voicing),
                         ("breathiness", breathiness), ("energy", energy)) if on]) or "none"})
print_table(summary, [("k", "setting"), ("v", "value")])

save_state(
    model_type=model_type,
    model_dir=str(MODEL_DIR),
    exp_name=EXP_NAME,
    config_path=str(CONFIG_PATH),
    binary_dir=str(BINARY_DIR),
    binary_on_drive=not binary_location.startswith("local"),
    training_mode=training_mode,
)
print("\n  Next: step 4 - binarize.")

In [ ]:
#@title 4. Binarize  -  pack the dataset for training
#@markdown Turns your dataset into the packed format training reads. Run once per dataset.

backup_binary_to_drive = False  #@param {type:"boolean"}
#@markdown <font size="-1">Only relevant when step 3 used local disk. Copies the result next to your
#@markdown model so a dead runtime does not cost you the binarize pass. Costs upload time and Drive space.</font>

show_full_log = True  #@param {type:"boolean"}

# --------------------------------------------------------------------------- #

import importlib
import shutil
import sys
import time
from pathlib import Path

sys.path.insert(0, "/content")
import dsnb
importlib.reload(dsnb)
from dsnb import (DEFAULT_ACCENT, DONE_ACCENT, FAST_ACCENT, LOG_DIR, REPO, die, header,
                  human_size, human_time, info, is_fast, load_state, ok, preflight,
                  print_table, step, warn)

state = load_state(required=["setup_done", "config_path", "binary_dir"])
PY = dsnb.training_python()
CONFIG = Path(state["config_path"])
BINARY_DIR = Path(state["binary_dir"])
MODEL_DIR = Path(state["model_dir"])
LOG = LOG_DIR / "binarize.log"

FAST = is_fast(state.get("training_mode", ""))
workers = (dsnb.resolve_merged_config(CONFIG).get("binarization_args") or {}).get("num_workers", 0)     if CONFIG.exists() else 0
header("Binarize",
       subtitle=(f"⚡ fast mode - {workers} parallel workers" if FAST
                 else "Original mode - single process"),
       accent=FAST_ACCENT if FAST else DEFAULT_ACCENT)

if not CONFIG.exists():
    die(f"config not found: {CONFIG}", fix="Re-run step 3.")

with step("Re-checking the config"):
    preflight(CONFIG, state["model_type"])

if BINARY_DIR.exists() and any(BINARY_DIR.iterdir()):
    warn(f"{BINARY_DIR} already has data - it will be overwritten")

started = time.time()
BINARIZE = [PY, "scripts/binarize.py", "--config", str(CONFIG), "--reset"]
with step("Running binarize.py"):
    code = dsnb.run(BINARIZE, cwd=REPO, log_file=LOG, quiet=not show_full_log, check=not FAST)
    if FAST and code != 0:
        # Parallel binarize is the experimental part. Rather than leave you stuck,
        # fall back to the stock single-process pass and say so.
        warn(f"parallel binarize failed (exit {code}) - retrying single-process")
        cfg = dsnb.load_yaml(CONFIG)
        cfg.setdefault("binarization_args", {})["num_workers"] = 0
        dsnb.dump_yaml(cfg, CONFIG)
        dsnb.run(BINARIZE, cwd=REPO, log_file=LOG, quiet=not show_full_log)
        ok("single-process binarize worked; the config now uses it")

# --------------------------------------------------------------------------- #
produced = sorted(p for p in BINARY_DIR.glob("*") if p.is_file())
if not produced:
    die("binarize finished but produced no files",
        fix=f"Check the log: {LOG}")

total = sum(p.stat().st_size for p in produced)
rows = [{"file": p.name, "size": human_size(p.stat().st_size)} for p in produced]
header(f"Binarized in {human_time(time.time() - started)}", accent=DONE_ACCENT)
print_table(rows, [("file", "file"), ("size", "size")])
print()
info(f"{BINARY_DIR}  ({human_size(total)})")

if backup_binary_to_drive and not state.get("binary_on_drive"):
    with step("Copying binary data to Drive"):
        target = MODEL_DIR / "binary"
        if target.exists():
            shutil.rmtree(target)
        shutil.copytree(BINARY_DIR, target)
        ok(f"backup at {target}")
        info("Step 5 will pick this up automatically if the local copy is gone.")

print("\n  Next: step 5 - train.")

---
## &#128640; Train

Training runs until `max_updates` or until you interrupt the cell. Checkpoints are
already on disk either way &mdash; interrupting loses nothing since the last save.

Colab will disconnect eventually. To pick up where you left off in a new session:
run cell 0, cell 1, then cell 5 with `resume_from_model_dir` pointing at your model
folder. Steps 2&ndash;4 are not needed.

> &#9889; **Fast mode on a T4:** keep `torch_compile` ticked. On a T4 it is the biggest single
> win, because the card spends most of its time moving data between small kernels and the
> compiler fuses them. The first few minutes are slower while it compiles. Fast mode also
> forces fp16 on a T4, so a `bf16-mixed` or `32-true` choice in step 3 cannot quietly run
> at a fraction of the speed.

In [ ]:
#@title 5. Train  -  or continue an earlier run
#@markdown Starts (or continues) training and opens TensorBoard above the log.

#@markdown ---
#@markdown ### Training mode
training_mode = "same as step 3"  #@param ["same as step 3", "Original", "Woah's Epic Faster Training (experimental)"]
#@markdown <font size="-1">&#9889; **Woah's Epic Faster Training** replaces `batch_size` and
#@markdown `max_batch_frames` below with values sized to this GPU's VRAM, picks the fastest precision
#@markdown for the card (fp16 on a T4, bf16 on L4/A100), fuses AdamW, keeps a CPU core free for the
#@markdown training loop and trims validation, logging and progress-bar redraws. Your schedule
#@markdown (`save_every`, `max_updates`, checkpoints kept) is never touched. **Original** uses the
#@markdown sliders as-is.</font>
torch_compile = True  #@param {type:"boolean"}
#@markdown <font size="-1">Fast mode only. Compiles the diffusion backbone with `torch.compile` - the
#@markdown single biggest speed-up on a T4, after a few minutes of compiling while the first steps run.
#@markdown If the compiler fails, training restarts without it automatically.</font>

#@markdown ---
#@markdown ### Schedule
batch_size = 16  #@param {type:"slider", min:1, max:64, step:1}
#@markdown <font size="-1">Too high gives CUDA out-of-memory; too low wastes the GPU. 16 suits a T4 at default sizes.</font>
max_batch_frames = 32000  #@param {type:"slider", min:4000, max:80000, step:1000}
save_every = 2000  #@param {type:"slider", min:200, max:10000, step:100}
max_updates = 160000  #@param {type:"slider", min:1000, max:1000000, step:1000}
keep_last_n_checkpoints = 3  #@param {type:"slider", min:1, max:10, step:1}
open_tensorboard = True  #@param {type:"boolean"}

#@markdown ---
#@markdown ### Continue an earlier run &nbsp;<font size="-1">(new runtime? just fill this in)</font>
resume_from_model_dir = ""  #@param {type:"string"}
#@markdown <font size="-1">Point at the model folder from a previous session. Steps 2 and 3 are not needed -
#@markdown it reads `notebook_config.yaml` that step 3 saved there.</font>
copy_binary_back_to_local = True  #@param {type:"boolean"}

# --------------------------------------------------------------------------- #

import importlib
import shlex
import shutil
import sys
from pathlib import Path

from IPython import get_ipython

sys.path.insert(0, "/content")
import dsnb
importlib.reload(dsnb)
from dsnb import (DEFAULT_ACCENT, DONE_ACCENT, FAST_ACCENT, FAST_ENV, LOG_DIR, ORIGINAL_MODE,
                  REPO, die, dump_yaml, find_latest_ckpt, header, human_size, info, is_fast,
                  link_experiment, load_state, load_yaml, ok, print_table,
                  sanitize_exp_name, save_state, step, warn)

state = load_state(required=["setup_done"])
PY = dsnb.training_python()

# --------------------------------------------------------------------------- #
#  Where are we training?
# --------------------------------------------------------------------------- #
if resume_from_model_dir.strip():
    MODEL_DIR = Path(resume_from_model_dir.strip().strip('"').strip("'"))
    if not MODEL_DIR.exists():
        die(f"model folder not found: {MODEL_DIR}",
            fix="Check the path, or leave this blank to train the model from step 3.")
    CONFIG = MODEL_DIR / "notebook_config.yaml"
    if not CONFIG.exists():
        candidate = MODEL_DIR / "config.yaml"
        if not candidate.exists():
            die(f"no config found in {MODEL_DIR}",
                fix="Expected notebook_config.yaml (written by step 3) or config.yaml\n"
                    "(written by a previous training run).")
        CONFIG = candidate
        warn("using config.yaml from the previous run")
    EXP_NAME = sanitize_exp_name(MODEL_DIR.name)
    MODEL_TYPE = "variance" if "predict_dur" in dsnb.resolve_merged_config(CONFIG) else "acoustic"
    info(f"resuming a {MODEL_TYPE} model from {MODEL_DIR}")
else:
    state = load_state(required=["config_path", "model_dir", "exp_name"])
    MODEL_DIR = Path(state["model_dir"])
    CONFIG = Path(state["config_path"])
    EXP_NAME = state["exp_name"]
    MODEL_TYPE = state["model_type"]

MODE = (state.get("training_mode", ORIGINAL_MODE) if training_mode == "same as step 3"
        else training_mode)
FAST = is_fast(MODE)
header(f"Train {EXP_NAME}",
       subtitle=("⚡ Woah's Epic Faster Training mode (experimental)" if FAST
                 else "Original training mode"),
       accent=FAST_ACCENT if FAST else DEFAULT_ACCENT)

with step("Preparing work directory"):
    link_experiment(EXP_NAME, MODEL_DIR)
    ok(f"checkpoints/{EXP_NAME} -> {MODEL_DIR}")


# --------------------------------------------------------------------------- #
#  Binary data - after a runtime restart the local copy is gone
# --------------------------------------------------------------------------- #
cfg = load_yaml(CONFIG)
binary_dir = Path(cfg.get("binary_data_dir", ""))

with step("Locating binarized data"):
    def has_data(path):
        return path.exists() and any(path.glob("*.data"))

    if has_data(binary_dir):
        ok(f"{binary_dir}")
    else:
        backup = MODEL_DIR / "binary"
        if has_data(backup):
            if copy_binary_back_to_local and str(binary_dir).startswith("/content/binary"):
                info("restoring the Drive backup to local disk (much faster to read from)")
                if binary_dir.exists():
                    shutil.rmtree(binary_dir)
                shutil.copytree(backup, binary_dir)
                ok(f"restored to {binary_dir}")
            else:
                cfg["binary_data_dir"] = str(backup)
                binary_dir = backup
                warn("training straight off Drive - expect slower steps")
        else:
            die("no binarized data found",
                fix=f"Looked in:\n  {binary_dir}\n  {backup}\n"
                    "Run steps 2-4 to rebuild it, or tick 'backup_binary_to_drive' in step 4\n"
                    "next time so a lost runtime does not cost you the binarize pass.")


# --------------------------------------------------------------------------- #
#  Apply the schedule
# --------------------------------------------------------------------------- #
with step("Applying training schedule"):
    cfg["val_check_interval"] = save_every
    cfg["max_updates"] = max_updates
    cfg["num_ckpt_keep"] = keep_last_n_checkpoints
    # The precision you chose in step 3 is remembered, so switching back to
    # Original after a fast run really is the original setup again.
    cfg.setdefault("nb_user_precision", cfg.get("pl_trainer_precision", "16-mixed"))
    stock = dsnb.resolve_merged_config(REPO / "configs" / f"{MODEL_TYPE}.yaml")
    if FAST:
        gpu = dsnb.gpu_specs()
        optimizer_cls = (dsnb.resolve_merged_config(CONFIG).get("optimizer_args")
                         or {}).get("optimizer_cls", "")
        tuned = dsnb.fast_train_overrides(MODEL_TYPE, gpu, optimizer_cls)
        dsnb.apply_training_mode(cfg, rows=tuned)
        if not gpu:
            warn("could not read the GPU - fast mode assumed a 16 GB T4")
    else:
        dsnb.apply_training_mode(cfg, stock=stock)
        cfg["pl_trainer_precision"] = cfg["nb_user_precision"]
        cfg["max_batch_size"] = batch_size
        cfg["max_batch_frames"] = max_batch_frames
    dump_yaml(cfg, CONFIG)
    ok(f"batch {cfg['max_batch_size']} / {cfg['max_batch_frames']} frames / "
       f"save every {save_every} / stop at {max_updates}")

if FAST:
    header("⚡ Fast-mode tuning", subtitle="stock DiffSinger settings, sized to this runtime",
           accent=FAST_ACCENT)
    print_table([{"k": k, "v": v, "why": why} for k, v, why in tuned]
                + [{"k": k, "v": v, "why": "environment"} for k, v in FAST_ENV.items()]
                + [{"k": "torch.compile", "v": "on" if torch_compile else "off",
                    "why": "backbone compiled, falls back to eager on failure"
                    if torch_compile else "ticked off in this cell"},
                   {"k": "TF32", "v": "on", "why": "fp32 matmuls / convs on Ampere+"}],
                [("k", "setting"), ("v", "value"), ("why", "why")])
    if torch_compile:
        info("The first steps are slow while torch.compile builds kernels - that is expected.")
    info("If the first steps fail with CUDA out-of-memory, re-run with training_mode = Original.")

existing = find_latest_ckpt(MODEL_DIR)
if existing:
    info(f"found {existing.name} - training continues from there")
else:
    info("no checkpoint yet - starting from step 0")

free = dsnb.disk_free("/content")
if free < 5 * 1024 ** 3:
    warn(f"only {human_size(free)} free on /content; checkpoints may fail to write")


# --------------------------------------------------------------------------- #
#  TensorBoard
# --------------------------------------------------------------------------- #
if open_tensorboard:
    tb_dir = MODEL_DIR / "lightning_logs"
    tb_dir.mkdir(parents=True, exist_ok=True)
    ipython = get_ipython()
    ipython.run_line_magic("reload_ext", "tensorboard")
    ipython.run_line_magic("tensorboard", f"--logdir {shlex.quote(str(tb_dir))}")


# --------------------------------------------------------------------------- #
header("Training", subtitle="interrupt this cell to stop - your checkpoints are already saved",
       accent=FAST_ACCENT if FAST else DEFAULT_ACCENT)
save_state(model_dir=str(MODEL_DIR), config_path=str(CONFIG), exp_name=EXP_NAME,
           model_type=MODEL_TYPE, training_mode=MODE)

TRAIN_LOG = LOG_DIR / "train.log"
TRAIN_ARGS = ["--config", str(CONFIG), "--exp_name", EXP_NAME, "--reset"]


def train(compile_backbone):
    """Run one training process; returns (exit code, this run's log text)."""
    start = TRAIN_LOG.stat().st_size if TRAIN_LOG.exists() else 0
    if FAST:
        # The launcher applies the runtime tweaks, then runs the stock train.py.
        cmd = [PY, dsnb.write_fast_launcher(), REPO] + TRAIN_ARGS
        env = {**FAST_ENV, "NB_TORCH_COMPILE": "1" if compile_backbone else "0"}
    else:
        cmd, env = [PY, "scripts/train.py"] + TRAIN_ARGS, None
    code = dsnb.run(cmd, cwd=REPO, log_file=TRAIN_LOG, check=False, env=env)
    with TRAIN_LOG.open("rb") as fh:
        fh.seek(start)
        return code, fh.read().decode("utf-8", errors="replace")


def ran_out_of_memory(log_text):
    return "CUDA out of memory" in log_text or "OutOfMemoryError" in log_text


exit_code, this_run = train(FAST and torch_compile)
# Compile errors dynamo recovers from still leave warnings in the log, so only a
# crash that was not an out-of-memory counts as the compiler's fault.
if (FAST and torch_compile and exit_code != 0 and not ran_out_of_memory(this_run)
        and any(marker in this_run for marker in dsnb.COMPILE_FAILURE_MARKERS)):
    # Checkpoints are already on disk, so the retry resumes rather than restarts.
    warn("torch.compile failed on this runtime - restarting training without it")
    exit_code, this_run = train(False)

# --exp_name is what DiffSinger turns into checkpoints/<exp_name>. The original
# notebook passed `${save_dir}`, which bash expands to an empty string, and then
# worked around the resulting mess by rewriting utils/hparams.py on every run.

latest = find_latest_ckpt(MODEL_DIR)
header("Training stopped", accent=DONE_ACCENT if latest else DEFAULT_ACCENT)
if exit_code != 0 and ran_out_of_memory(this_run):
    warn("the GPU ran out of memory.")
    if FAST:
        warn("fast mode's batch size is too big for this dataset - set training_mode to")
        warn("Original (or keep fast mode on a bigger GPU) and run this cell again.")
    else:
        warn("lower batch_size / max_batch_frames above and run this cell again.")
if latest:
    ok(f"latest checkpoint: {latest.name} ({human_size(latest.stat().st_size)})")
    print(f"\n  Model folder: {MODEL_DIR}")
    print("  Next: step 6 - export to ONNX.")
else:
    warn("no checkpoint was written - scroll up for the reason")

---
## &#128230; Export and package

Step 6 converts checkpoints to ONNX; step 7 assembles them into an OpenUtau voicebank.

A complete voicebank needs **both** an acoustic and a variance model. Acoustic alone
renders audio but cannot predict duration, pitch or parameters on its own.

In [ ]:
#@title 6. Export to ONNX
#@markdown Converts trained checkpoints into the ONNX files OpenUtau loads.
#@markdown Fill in whichever models you have - blanks are skipped.

acoustic_model_dir = ""  #@param {type:"string"}
variance_model_dir = ""  #@param {type:"string"}
output_dir = ""  #@param {type:"string"}
#@markdown <font size="-1">An `onnx/acoustic` and `onnx/variance` folder are created here.</font>

export_backend = "isolated python 3.10 (recommended)"  #@param ["isolated python 3.10 (recommended)", "current runtime"]
#@markdown <font size="-1">DiffSinger pins `torch==1.13.1` for ONNX export, which needs Python 3.10.
#@markdown The isolated option builds a small CPU-only 3.10 environment in about a minute.
#@markdown <br>The original notebook installed Miniconda over `/usr/local` instead, which replaced
#@markdown the whole interpreter, forced a runtime restart, and wiped every variable the later cells needed.</font>

show_full_log = False  #@param {type:"boolean"}

# --------------------------------------------------------------------------- #

import hashlib
import importlib
import shutil
import sys
from pathlib import Path

sys.path.insert(0, "/content")
import dsnb
importlib.reload(dsnb)
from dsnb import (LOG_DIR, REPO, die, find_latest_ckpt, header, human_size, info,
                  link_experiment, load_state, ok, print_table, run, sanitize_exp_name,
                  save_state, step, warn)

state = load_state(required=["setup_done"])
LOG = LOG_DIR / "export.log"
QUIET = not show_full_log
ENV_DIR = dsnb.ONNX_ENV
ENV_PY = ENV_DIR / "bin" / "python"

header("Export ONNX")


def clean_path(value):
    return value.strip().strip('"').strip("'")


jobs = []
for kind, raw in (("acoustic", acoustic_model_dir), ("variance", variance_model_dir)):
    if not clean_path(raw):
        continue
    folder = Path(clean_path(raw))
    if not folder.exists():
        die(f"{kind} model folder not found: {folder}",
            fix="Point at the folder that holds model_ckpt_steps_*.ckpt and config.yaml.")
    ckpt = find_latest_ckpt(folder)
    if ckpt is None:
        die(f"no *.ckpt in {folder}",
            fix="Give the folder containing the checkpoints, not the .ckpt file itself.")
    if not (folder / "config.yaml").exists():
        die(f"no config.yaml in {folder}",
            fix="Training writes config.yaml next to the checkpoints. Without it the\n"
                "exporter cannot know how the model was built.")
    jobs.append({"kind": kind, "dir": folder, "ckpt": ckpt,
                 "exp": sanitize_exp_name(folder.name)})

if not jobs:
    die("nothing to export",
        fix="Fill in acoustic_model_dir and/or variance_model_dir.")
if not clean_path(output_dir):
    die("output_dir is empty", fix="e.g. /content/drive/MyDrive/DiffSinger/my_singer")
OUT_ROOT = Path(clean_path(output_dir))

for job in jobs:
    info(f"{job['kind']}: {job['ckpt'].name} from {job['dir'].name}")


# --------------------------------------------------------------------------- #
#  Export environment
# --------------------------------------------------------------------------- #
# Plain PyPI, not the +cpu index. The CPU wheels are a smaller download, but they
# live on a separate index, and mixing indexes needs --index-strategy juggling that
# is far more likely to go wrong than a one-off 900 MB fetch is to be slow.
EXPORT_TORCH = ["torch==1.13.1", "torchvision==0.14.1", "torchaudio==0.13.1"]
EXPORT_IMPORTS = [
    ("torch", "torch"),
    ("lightning", "lightning"),
    ("numpy", "numpy"),
    ("librosa", "librosa"),
    ("torchmetrics", "torchmetrics"),
    ("onnx", "onnx"),
    ("onnxruntime", "onnxruntime"),
    ("onnxsim", "onnxsim"),
    ("yaml", "PyYAML"),
    ("pkg_resources", "setuptools"),
]

if export_backend.startswith("isolated"):
    with step(f"Export environment (python {dsnb.ONNX_ENV_PYTHON_VERSION})"):
        requirements = []
        for raw in (REPO / "requirements-onnx.txt").read_text(encoding="utf-8").splitlines():
            entry = raw.split("#", 1)[0].strip()
            if entry and not entry.lower().startswith(("torch", "torchvision", "torchaudio")):
                requirements.append(entry)

        # Keyed to the requirement list, not just "does the folder exist" - an env
        # built by an older version of this notebook has to be rebuilt, not reused.
        fingerprint = hashlib.sha256(
            "|".join([dsnb.ONNX_ENV_PYTHON_VERSION] + EXPORT_TORCH + requirements).encode()
        ).hexdigest()[:16]
        marker = ENV_DIR.with_name(".onnx_env_complete")
        usable = (ENV_PY.exists()
                  and marker.exists()
                  and marker.read_text(encoding="utf-8").strip() == fingerprint
                  and not dsnb.module_versions(str(ENV_PY), EXPORT_IMPORTS)[1])

        if usable:
            ok("already built and everything imports - reusing it")
        else:
            if ENV_PY.exists():
                info("rebuilding: the existing environment is stale or incomplete")
                shutil.rmtree(ENV_DIR)
            marker.unlink(missing_ok=True)
            print(f"  torch 1.13.1 + {len(requirements)} packages, one resolve")
            info("torchmetrics is unpinned upstream and its recent versions demand")
            info("torch>=2, so torch has to be pinned inside the same resolve or it")
            info("gets silently upgraded and export.py dies on a missing symbol.")
            dsnb.build_env(
                ENV_DIR,
                dsnb.ONNX_ENV_PYTHON_VERSION,
                EXPORT_TORCH + requirements,
                log_file=LOG,
                quiet=QUIET,
            )
            dsnb.assert_pinned(ENV_PY, "torch", "1.13.1", log_file=LOG)
            rows, broken = dsnb.module_versions(str(ENV_PY), EXPORT_IMPORTS)
            if broken:
                print_table(rows, [("module", "module"), ("version", "version")])
                for module, reason in broken:
                    warn(f"{module}: {reason}")
                die(f"{len(broken)} module(s) missing from the export environment",
                    fix=f"Install them and re-run:\n"
                        f"  !{dsnb.uv_path()} pip install --python {ENV_PY} "
                        + " ".join(dict(EXPORT_IMPORTS)[m] for m, _ in broken))
            marker.write_text(fingerprint, encoding="utf-8")
            ok(f"built and verified at {ENV_DIR}")
    PYTHON = str(ENV_PY)
else:
    PYTHON = dsnb.training_python()
    warn("using the training environment - export.py pins torch 1.13, so if it")
    warn("complains, switch back to 'isolated python 3.10' and re-run.")


# --------------------------------------------------------------------------- #
#  Export
# --------------------------------------------------------------------------- #
# What the exporter writes is `{exp_name}.<thing>`; OpenUtau wants fixed names.
# Because we know exp_name exactly, this is a rename table rather than the
# substring guessing the original notebook did (which left `*.phonemes.json`
# untouched and quietly broke the voicebank builder further down).
RENAMES = {
    "acoustic": {
        "onnx": "acoustic.onnx",
        "phonemes.json": "acoustic_phonemes.json",
        "languages.json": "acoustic_languages.json",
    },
    "variance": {
        "linguistic.onnx": "linguistic.onnx",
        "dur.onnx": "dur.onnx",
        "pitch.onnx": "pitch.onnx",
        "variance.onnx": "variance.onnx",
        "phonemes.json": "variance_phonemes.json",
        "languages.json": "variance_languages.json",
    },
}

results = []
for job in jobs:
    kind, folder, exp = job["kind"], job["dir"], job["exp"]
    out = OUT_ROOT / "onnx" / kind
    if out.exists():
        shutil.rmtree(out)
    out.mkdir(parents=True, exist_ok=True)

    with step(f"Exporting {kind}"):
        link_experiment(exp, folder)
        run([PYTHON, "scripts/export.py", kind, "--exp", exp, "--out", str(out)],
            cwd=REPO, log_file=LOG, quiet=QUIET)

    with step(f"Normalising {kind} filenames"):
        prefix = exp + "."
        renamed = []
        for path in sorted(out.iterdir()):
            if not path.name.startswith(prefix):
                continue
            tail = path.name[len(prefix):]
            if tail.endswith(".emb"):
                new_name = tail                      # {speaker}.emb
            else:
                new_name = RENAMES[kind].get(tail)
            if not new_name:
                continue
            path.rename(out / new_name)
            renamed.append(f"{path.name} -> {new_name}")
        for line in renamed:
            ok(line)
        if not renamed:
            warn("no files matched the expected naming - check the export log")

        # The voicebank builder reads the training config for extra phonemes and
        # audio settings, so keep a copy right next to the ONNX files.
        shutil.copy(folder / "config.yaml", out / "config.yaml")
        for dictionary in (folder / "dictionaries").glob("*.txt"):
            shutil.copy(dictionary, out / dictionary.name)
        ok("config.yaml and dictionaries copied alongside the ONNX files")

    files = sorted(p for p in out.iterdir() if p.is_file())
    results += [{"model": kind, "file": p.name, "size": human_size(p.stat().st_size)}
                for p in files]
    job["out"] = str(out)


header("Export complete", accent=dsnb.DONE_ACCENT)
print_table(results, [("model", "model"), ("file", "file"), ("size", "size")])

expected = {"acoustic": ["acoustic.onnx", "acoustic_phonemes.json", "acoustic_languages.json",
                         "dsconfig.yaml"],
            "variance": ["linguistic.onnx", "dsconfig.yaml"]}
for job in jobs:
    missing = [name for name in expected[job["kind"]]
               if not (Path(job["out"]) / name).exists()]
    if missing:
        warn(f"{job['kind']} is missing: {', '.join(missing)}")

save_state(
    onnx_acoustic=next((j["out"] for j in jobs if j["kind"] == "acoustic"), None),
    onnx_variance=next((j["out"] for j in jobs if j["kind"] == "variance"), None),
)
print("\n  Next: step 7 - build the OpenUtau voicebank.")

In [ ]:
#@title 7. Build OpenUtau voicebank
#@markdown Assembles the exported ONNX files into an installable OpenUtau voicebank and zips it.
#@markdown <br><font size="-1">Leave the paths blank to reuse whatever step 6 just exported.</font>

voicebank_name = ""  #@param {type:"string"}
acoustic_onnx_folder = ""  #@param {type:"string"}
variance_onnx_folder = ""  #@param {type:"string"}
save_to = ""  #@param {type:"string"}
custom_vocoder_onnx = ""  #@param {type:"string"}
#@markdown <font size="-1">Path to a fine-tuned vocoder `.onnx` (its `vocoder.yaml` must sit beside it). Leave blank for the standard one.</font>
make_zip = True  #@param {type:"boolean"}

# --------------------------------------------------------------------------- #

import importlib
import shutil
import sys
from pathlib import Path

sys.path.insert(0, "/content")
import dsnb
importlib.reload(dsnb)
from dsnb import (die, dump_yaml, header, human_size, info, load_state, load_yaml, ok,
                  print_table, sanitize_exp_name, step, warn)

state = load_state()

# OpenUtau classifies every symbol; a wrong guess only affects automatic
# phoneme timing hints, never whether the bank loads.
PHONEME_TYPES = {
    "a": "vowel", "i": "vowel", "u": "vowel", "e": "vowel", "o": "vowel",
    "A": "vowel", "I": "vowel", "U": "vowel", "E": "vowel", "O": "vowel",
    "N": "vowel", "M": "vowel",
    "b": "stop", "d": "stop", "g": "stop", "k": "stop", "p": "stop",
    "q": "stop", "t": "stop",
    "c": "affricate", "j": "affricate",
    "f": "fricative", "h": "fricative", "s": "fricative", "v": "fricative",
    "z": "fricative",
    "l": "liquid", "r": "liquid",
    "m": "nasal", "n": "nasal",
    "w": "semivowel", "y": "semivowel",
}
# OpenUtau's built-in phonemizers already understand these language codes, so a
# dsdict for them does not need a replacements table.
OU_CANON_LANGUAGES = {"zh", "en", "de", "it", "ja", "yue", "ko", "pt", "ru", "es"}

header("Build OpenUtau voicebank")


def clean(value):
    return value.strip().strip('"').strip("'")


ACOUSTIC = Path(clean(acoustic_onnx_folder) or state.get("onnx_acoustic") or "")
VARIANCE = Path(clean(variance_onnx_folder) or state.get("onnx_variance") or "")
if not clean(voicebank_name):
    die("voicebank_name is empty", fix="This is the name OpenUtau shows in the singer list.")
NAME = clean(voicebank_name)
FOLDER_NAME = sanitize_exp_name(NAME.replace(" ", ""))
if not clean(save_to):
    die("save_to is empty", fix="e.g. /content/drive/MyDrive/DiffSinger")
DEST = Path(clean(save_to))

if not ACOUSTIC or not ACOUSTIC.exists():
    die(f"acoustic ONNX folder not found: {ACOUSTIC}",
        fix="Run step 6 first, or point at the folder containing acoustic.onnx.")
if not (ACOUSTIC / "acoustic.onnx").exists():
    die(f"no acoustic.onnx in {ACOUSTIC}",
        fix="Step 6 renames the export to acoustic.onnx. If you exported by hand,\n"
            "rename `<exp>.onnx` to `acoustic.onnx` and its `.phonemes.json` /\n"
            "`.languages.json` to `acoustic_phonemes.json` / `acoustic_languages.json`.")

has_variance_folder = bool(VARIANCE) and VARIANCE.exists()
if not has_variance_folder:
    warn("no variance model - the bank will render audio but will not predict")
    warn("duration, pitch or parameters on its own.")

parts = {
    "linguistic": has_variance_folder and (VARIANCE / "linguistic.onnx").exists(),
    "dur": has_variance_folder and (VARIANCE / "dur.onnx").exists(),
    "pitch": has_variance_folder and (VARIANCE / "pitch.onnx").exists(),
    "variance": has_variance_folder and (VARIANCE / "variance.onnx").exists(),
}
info("components found: " + (", ".join(k for k, v in parts.items() if v) or "acoustic only"))

BUILD = DEST / FOLDER_NAME
if BUILD.exists():
    shutil.rmtree(BUILD)

DSMAIN = BUILD / "dsmain"
SUBFOLDERS = {"dur": BUILD / "dsdur", "pitch": BUILD / "dspitch",
              "variance": BUILD / "dsvariance"}


# --------------------------------------------------------------------------- #
#  1. Copy files
# --------------------------------------------------------------------------- #
with step("Laying out the bank"):
    (DSMAIN / "embeds").mkdir(parents=True, exist_ok=True)
    for filename in ("acoustic.onnx", "acoustic_phonemes.json", "acoustic_languages.json"):
        source = ACOUSTIC / filename
        if not source.exists():
            die(f"{filename} is missing from {ACOUSTIC}",
                fix="Re-run step 6; the export did not finish.")
        shutil.copy(source, DSMAIN / filename)
    shutil.copy(ACOUSTIC / "dsconfig.yaml", BUILD / "dsconfig.yaml")

    acoustic_embeds = []
    for emb in sorted(ACOUSTIC.glob("*.emb")):
        shutil.copy(emb, DSMAIN / "embeds" / emb.name)
        acoustic_embeds.append(emb.stem)

    if parts["linguistic"]:
        shutil.copy(VARIANCE / "linguistic.onnx", DSMAIN / "linguistic.onnx")

    variance_embeds = []
    for key, folder in SUBFOLDERS.items():
        if not parts[key]:
            continue
        (folder / "embeds").mkdir(parents=True, exist_ok=True)
        shutil.copy(VARIANCE / f"{key}.onnx", folder / f"{key}.onnx")
        for emb in sorted(VARIANCE.glob("*.emb")):
            shutil.copy(emb, folder / "embeds" / emb.name)
        variance_embeds = [emb.stem for emb in sorted(VARIANCE.glob("*.emb"))]
    ok(f"{len(acoustic_embeds)} acoustic embed(s), {len(variance_embeds)} variance embed(s)")


# --------------------------------------------------------------------------- #
#  2. Configs
# --------------------------------------------------------------------------- #
with step("Writing configs"):
    acoustic_train_cfg = load_yaml(ACOUSTIC / "config.yaml") if (
        ACOUSTIC / "config.yaml").exists() else {}
    variance_train_cfg = load_yaml(VARIANCE / "config.yaml") if (
        has_variance_folder and (VARIANCE / "config.yaml").exists()) else {}
    variance_dsconfig = load_yaml(VARIANCE / "dsconfig.yaml") if (
        has_variance_folder and (VARIANCE / "dsconfig.yaml").exists()) else {}

    # Name the vocoder the model was actually trained against rather than assuming.
    trained_vocoder = Path(acoustic_train_cfg.get("vocoder_ckpt", "")).parent.name
    main_config = load_yaml(BUILD / "dsconfig.yaml")
    main_config.update({
        "singer_type": "diffsinger",
        "acoustic": "dsmain/acoustic.onnx",
        "phonemes": "dsmain/acoustic_phonemes.json",
        "languages": "dsmain/acoustic_languages.json",
        "vocoder": trained_vocoder or "pc_nsf_hifigan_44.1k_hop512_128bin_2025.02",
    })
    if acoustic_embeds:
        main_config["speakers"] = [f"embeds/{name}" for name in acoustic_embeds]
    dump_yaml(main_config, BUILD / "dsconfig.yaml")

    # Shared values for the sub-model configs. Everything is read from the two
    # training configs, and every lookup has a default - the original crashed
    # with a NameError whenever a bank had pitch.onnx but no variance.onnx.
    shared = {
        "phonemes": "../dsmain/acoustic_phonemes.json",
        "languages": "../dsmain/acoustic_languages.json",
        "linguistic": "../dsmain/linguistic.onnx",
        "sample_rate": acoustic_train_cfg.get("audio_sample_rate", 44100),
        "hop_size": acoustic_train_cfg.get("hop_size", 512),
        "hidden_size": variance_dsconfig.get(
            "hidden_size", variance_train_cfg.get("hidden_size", 256)),
        "use_lang_id": bool(acoustic_train_cfg.get("use_lang_id", False)),
        "use_continuous_acceleration": variance_dsconfig.get(
            "use_continuous_acceleration", True),
    }
    if variance_embeds:
        shared["speakers"] = [f"embeds/{name}" for name in variance_embeds]

    if parts["dur"]:
        dump_yaml({**shared, "dur": "dur.onnx", "predict_dur": True},
                  SUBFOLDERS["dur"] / "dsconfig.yaml")
    if parts["variance"]:
        dump_yaml({
            **shared,
            "variance": "variance.onnx",
            "predict_dur": bool(variance_train_cfg.get("predict_dur", True)),
            "predict_tension": bool(variance_train_cfg.get("predict_tension", False)),
            "predict_voicing": bool(variance_train_cfg.get("predict_voicing", False)),
            "predict_energy": bool(variance_train_cfg.get("predict_energy", False)),
            "predict_breathiness": bool(variance_train_cfg.get("predict_breathiness", False)),
        }, SUBFOLDERS["variance"] / "dsconfig.yaml")
    if parts["pitch"]:
        dump_yaml({
            **shared,
            "pitch": "pitch.onnx",
            "use_expr": True,
            "predict_dur": bool(variance_train_cfg.get("predict_dur", True)),
            "use_note_rest": bool(
                variance_dsconfig.get("use_note_rest",
                                      variance_train_cfg.get("use_melody_encoder", False))),
        }, SUBFOLDERS["pitch"] / "dsconfig.yaml")
    ok("dsconfig.yaml written for " + ", ".join(
        ["dsmain"] + [f"ds{k}" for k, v in parts.items() if v and k != "linguistic"]))

    # character.txt / character.yaml
    (BUILD / "character.txt").write_text(
        f"name={NAME}\nauthor=\nweb=\nversion=\n", encoding="utf-8")
    character = {
        "name": NAME,
        "singer_type": "diffsinger",
        "text_file_encoding": "utf-8",
        "default_phonemizer": "OpenUtau.Core.DiffSinger.DiffSingerPhonemizer",
        "image": None,
        "portrait": None,
        "portrait_opacity": 0.45,
    }
    if acoustic_embeds:
        character["subbanks"] = [
            {"color": f"{index:02d}: {name}", "suffix": f"embeds/{name}"}
            for index, name in enumerate(acoustic_embeds, start=1)
        ]
    dump_yaml(character, BUILD / "character.yaml")
    ok("character.txt / character.yaml written")


# --------------------------------------------------------------------------- #
#  3. Dictionaries
# --------------------------------------------------------------------------- #
with step("Generating dsdict files"):
    dictionary_files = sorted(ACOUSTIC.glob("dictionary-*.txt"))
    if not dictionary_files:
        dictionary_files = sorted(ACOUSTIC.glob("*.txt"))
    if not dictionary_files:
        die("no phoneme dictionary found next to the acoustic ONNX",
            fix="Step 6 copies dictionary-*.txt from your model folder. Copy the\n"
                "dictionary-<lang>.txt files from your model folder into\n"
                f"{ACOUSTIC} and run this cell again.")

    extra_phonemes = acoustic_train_cfg.get("extra_phonemes") or []
    general_extra, per_language_extra = [], {}
    for phoneme in extra_phonemes:
        if "/" in str(phoneme):
            language, base = str(phoneme).split("/", 1)
            per_language_extra.setdefault(language, []).append(base)
        elif phoneme not in ("AP", "SP") and "trash" not in str(phoneme):
            general_extra.append(phoneme)

    multi_language = len(dictionary_files) > 1

    def language_of(path):
        stem = path.stem
        return stem.split("-", 1)[1] if "-" in stem else "custom"

    def symbol_type(phoneme):
        return PHONEME_TYPES.get(phoneme[0], "stop")

    def write_dsdict(path, symbols, entries, replacements=None):
        payload = {
            "symbols": [{"symbol": "SP", "type": "vowel"},
                        {"symbol": "AP", "type": "vowel"}] + symbols,
        }
        if replacements:
            payload["replacements"] = replacements
        payload["entries"] = entries + [{"grapheme": "SP", "phonemes": ["SP"]},
                                        {"grapheme": "AP", "phonemes": ["AP"]}]
        dump_yaml(payload, path)

    all_symbols, all_entries = [], []
    for phoneme in general_extra:
        all_symbols.append({"symbol": phoneme, "type": symbol_type(phoneme)})
        all_entries.append({"grapheme": phoneme, "phonemes": [phoneme]})

    per_language = {}
    for path in dictionary_files:
        language = language_of(path)
        phonemes = []
        for line in path.read_text(encoding="utf-8").splitlines():
            parts_of_line = line.split()
            if parts_of_line:
                phonemes.append(parts_of_line[0])
        phonemes += per_language_extra.get(language, [])
        per_language[language] = phonemes
        for phoneme in phonemes:
            symbol = f"{language}/{phoneme}" if multi_language else phoneme
            all_symbols.append({"symbol": symbol, "type": symbol_type(phoneme)})
            all_entries.append({"grapheme": phoneme, "phonemes": [symbol]})

    written = []
    if multi_language:
        for language, phonemes in per_language.items():
            symbols = [{"symbol": f"{language}/{p}", "type": symbol_type(p)} for p in phonemes]
            entries = [{"grapheme": p, "phonemes": [f"{language}/{p}"]} for p in phonemes]
            replacements = None
            if language not in OU_CANON_LANGUAGES:
                replacements = [{"from": p, "to": f"{language}/{p}"} for p in phonemes]
            # Every per-language dict carries the full symbol table so a phoneme
            # typed by hand from another language still resolves.
            write_dsdict(BUILD / f"dsdict-{language}.yaml", all_symbols, entries, replacements)
            written.append(f"dsdict-{language}.yaml ({len(phonemes)} phonemes)")
    write_dsdict(BUILD / "dsdict.yaml", all_symbols, all_entries)
    written.append(f"dsdict.yaml ({len(all_symbols)} symbols)")

    # Copy every dsdict into the sub-model folders as well. OpenUtau versions
    # differ on where they look; duplicating a few KB removes the guesswork.
    for source in sorted(BUILD.glob("dsdict*.yaml")):
        for key, folder in SUBFOLDERS.items():
            if parts[key]:
                shutil.copy(source, folder / source.name)
    for line in written:
        ok(line)


# --------------------------------------------------------------------------- #
#  4. Optional custom vocoder
# --------------------------------------------------------------------------- #
if clean(custom_vocoder_onnx):
    with step("Adding custom vocoder"):
        vocoder = Path(clean(custom_vocoder_onnx))
        if not vocoder.exists():
            die(f"vocoder not found: {vocoder}")
        vocoder_yaml = vocoder.parent / "vocoder.yaml"
        if not vocoder_yaml.exists():
            die(f"vocoder.yaml not found beside {vocoder.name}",
                fix="A custom vocoder needs both the .onnx and its vocoder.yaml.")
        target = BUILD / "dsvocoder"
        target.mkdir(parents=True, exist_ok=True)
        shutil.copy(vocoder, target / vocoder.name)
        shutil.copy(vocoder_yaml, target / "vocoder.yaml")
        main_config = load_yaml(BUILD / "dsconfig.yaml")
        main_config["vocoder"] = vocoder.stem
        dump_yaml(main_config, BUILD / "dsconfig.yaml")
        ok(f"dsvocoder/{vocoder.name}")


# --------------------------------------------------------------------------- #
#  5. Package
# --------------------------------------------------------------------------- #
tree = []
for path in sorted(BUILD.rglob("*")):
    if path.is_file():
        tree.append({"file": str(path.relative_to(BUILD)),
                     "size": human_size(path.stat().st_size)})

header(f"Voicebank built: {BUILD}", accent=dsnb.DONE_ACCENT)
print_table(tree, [("file", "file"), ("size", "size")])

if make_zip:
    with step("Zipping"):
        archive = shutil.make_archive(str(DEST / FOLDER_NAME), "zip",
                                      root_dir=str(DEST), base_dir=FOLDER_NAME)
        ok(f"{archive} ({human_size(Path(archive).stat().st_size)})")

print("\n  Install it by dropping the zip onto OpenUtau, or unzip it into")
print("  OpenUtau/Singers/. Enable the DiffSinger phonemizer if it is not already the default.")

---
## &#129520; Utilities

Optional. Neither is part of the normal train &rarr; export path.

In [ ]:
#@title Utility: drop speakers from a checkpoint
#@markdown Trims a multi-speaker checkpoint down to the speakers you want to publish.
#@markdown Run it on the **acoustic and the variance** checkpoint, then export those.

checkpoint_path = ""  #@param {type:"string"}
#@markdown <font size="-1">Path to a single `model_ckpt_steps_*.ckpt`, or to the folder holding it.</font>
keep_speaker_ids = ""  #@param {type:"string"}
#@markdown <font size="-1">IDs to **keep**, comma separated, e.g. `0,1`. Open `spk_map.json` in the
#@markdown model folder to see them: `{"natural": 0, "power": 1, "silly": 2}`.</font>
fill_dropped_embeds = "zeros"  #@param ["zeros", "random", "mean", "cyclic"]

# --------------------------------------------------------------------------- #

import importlib
import json
import sys
from pathlib import Path

sys.path.insert(0, "/content")
import dsnb
importlib.reload(dsnb)
from dsnb import REPO, die, find_latest_ckpt, header, human_size, info, ok, run, step

PY = dsnb.training_python()

header("Drop speakers")

raw = checkpoint_path.strip().strip('"').strip("'")
if not raw:
    die("checkpoint_path is empty")
source = Path(raw)
if source.is_dir():
    found = find_latest_ckpt(source)
    if found is None:
        die(f"no *.ckpt inside {source}")
    source = found
    info(f"using {source.name}")
if not source.exists():
    die(f"checkpoint not found: {source}")

keep = [part.strip() for part in keep_speaker_ids.split(",") if part.strip()]
if not keep:
    die("keep_speaker_ids is empty",
        fix="List the IDs you want to keep, e.g. 0,1")
if not all(part.isdigit() for part in keep):
    die(f"keep_speaker_ids must be numbers, got: {keep_speaker_ids}",
        fix='Use the numeric IDs from spk_map.json, e.g. 0,2 - not the names.')

spk_map = source.parent / "spk_map.json"
if spk_map.exists():
    mapping = json.loads(spk_map.read_text(encoding="utf-8"))
    inverted = {str(value): name for name, value in mapping.items()}
    unknown = [i for i in keep if i not in inverted]
    if unknown:
        die("no such speaker id: " + ", ".join(unknown),
            fix="spk_map.json says: " + json.dumps(mapping))
    info("keeping: " + ", ".join(f"{i} ({inverted[i]})" for i in keep))

destination = source.with_name(source.stem + "_spk-dropped.ckpt")
with step("Rewriting checkpoint"):
    run([PY, "scripts/drop_spk.py", str(source), str(destination),
         "--retain", ",".join(keep), "--fill", fill_dropped_embeds],
        cwd=REPO, quiet=True)

ok(f"{destination}  ({human_size(destination.stat().st_size)})")
info("Move this into a folder with the model's config.yaml before exporting it.")

In [ ]:
#@title Utility: package the prepared dataset
#@markdown Zips whatever step 2 produced so you can keep it and skip segmentation next time.
#@markdown Feed the zip back in as `csv + wav (DiffSinger)`.

save_to = ""  #@param {type:"string"}
#@markdown <font size="-1">Folder to write `<name>.zip` into, e.g. `/content/drive/MyDrive/DiffSinger`</font>
zip_name = "prepared_dataset"  #@param {type:"string"}

# --------------------------------------------------------------------------- #

import importlib
import shutil
import sys
from pathlib import Path

sys.path.insert(0, "/content")
import dsnb
importlib.reload(dsnb)
from dsnb import (dataset_summary, die, header, human_size, info, load_state, ok,
                  print_table, step)

state = load_state(required=["data_dir"])
DATA_DIR = Path(state["data_dir"])

header("Package prepared dataset")

if not DATA_DIR.exists():
    die(f"{DATA_DIR} is gone", fix="Run step 2 again to rebuild it.")
if not save_to.strip():
    die("save_to is empty", fix="e.g. /content/drive/MyDrive/DiffSinger")

DEST = Path(save_to.strip().strip('"').strip("'"))
DEST.mkdir(parents=True, exist_ok=True)

rows = dataset_summary(DATA_DIR)
for row in rows:
    row["hours"] = f"{row['hours']:.2f}"
print_table(rows, [("speaker", "speaker"), ("segments", "segments"), ("hours", "hours")])

with step("Zipping"):
    # root_dir=DATA_DIR keeps the speaker folders at the top of the archive, which
    # is exactly the shape step 2 expects to read back in. The original zipped
    # absolute /content paths, so the result could not be re-imported.
    archive = shutil.make_archive(str(DEST / zip_name.strip()), "zip", root_dir=str(DATA_DIR))

ok(f"{archive}  ({human_size(Path(archive).stat().st_size)})")
info("Re-import it in step 2 with data_type = 'csv + wav (DiffSinger)'.")

---
## &#128221; What is different from the original notebook

**Two training modes**

Original keeps the stock DiffSinger behaviour. Woah's Epic Faster Training (experimental)
parallelises binarize, sizes batches to the GPU, uses bf16 where the hardware has it, keeps
the GPU fed with more dataloader workers and trims validation &mdash; all through stock config
keys, so switching back is one dropdown.

**Setup went from ~1 hour to a few minutes**

| | Original | Here |
|---|---|---|
| Python | installed into Colab's interpreter, now 3.13, where `numpy<2`, `onnx~=1.16` and `onnxsim` have **no wheels** &mdash; pip silently falls back to compiling them from source on two cores | a Python 3.11 environment built by `uv`, where every pin has a wheel; the kernel is left alone |
| PyTorch | reinstalled from PyPI over Colab's CUDA build, often landing a CPU-only wheel | pinned `torch==2.5.1` (before `torch.load` flipped to `weights_only=True` and broke Lightning 2.3 checkpoint loading) |
| SOME deps | installed `SOME/requirements.txt`: `fairseq==0.12.2`, `gradio`, and `onnx==1.14.0` fighting DiffSinger's `onnx~=1.16.0` | skipped; SOME's script only imports `click`, `librosa`, `tqdm`, `yaml`, all already required |
| pip runs | 7 separate invocations, 7 dependency resolves | 1 resolve, all wheels |
| Downloads | ~1.5 GB serially over single-connection `wget` | `aria2c`, 4 files at a time, 16 connections each |
| ONNX export | installed Miniconda over `/usr/local`, replacing the interpreter and forcing a restart that wiped every variable | a CPU-only Python 3.10 venv built on demand in about a minute; your runtime is untouched |

Why two environments: training wants a modern torch, but `requirements-onnx.txt` pins
`torch==1.13.1`, which only exists for Python 3.10 or older. Rather than destroy the
runtime to satisfy both, each gets its own interpreter and the notebook calls into
them. Nothing is ever installed into the kernel except `uv` itself.

**Bugs that made things fail for real**

- `hnsep` was unpacked to `checkpoints/hnsep_240512/` while every config asks for
  `checkpoints/vr/model.pt`, so tension / voicing / breathiness extraction could
  never find its model. Now installed where the config points.
- The SOME checkpoint was looked up as `0119_continuous256_5spk` while the download
  is `0119_continuous128_5spk`. Now located by search, not by a hard-coded name.
- Label normalisation (`SP`&rarr;`pau`, `br`&rarr;`AP`) ran on the csv/ds branch instead of
  the lab branch &mdash; a comment in the original admits it &mdash; so it never touched a
  `.lab` file. It also used a raw string replace that corrupted any phoneme
  containing those letters. Now it rewrites the phoneme column only.
- Choosing `parselmouth` vs `harvest` for MIDI estimation did nothing: the choice
  was stored and never passed to the segmenter. Now passed as `-p`.
- Fine-tuning from your own base model raised `NameError: finetune_ckpt_path` &mdash; it
  was only assigned on the download branch.
- `--exp_name ${save_dir}` is bash expansion of a variable that does not exist, so
  the experiment name was always empty. The original compensated by rewriting
  `utils/hparams.py` and `utils/training_utils.py` with regexes on every run, which
  broke resuming and checkpoint deletion. DiffSinger resolves its work dir as
  `checkpoints/<exp_name>`, so a symlink does the whole job &mdash; no source patching.
- The ONNX exporter writes `<exp>.phonemes.json`, but the voicebank builder copied
  `acoustic_phonemes.json`. The failure was swallowed by a bare `except` and you got
  a voicebank that silently would not load. Filenames are now normalised explicitly,
  and nothing is caught and ignored.
- A bank with `pitch.onnx` but no `variance.onnx` hit `NameError: predict_dur`.
- Only the first row of each `transcriptions.csv` got its leading `SP` turned into
  `AP`; every other segment kept the wrong token. Now all of them.
- `!cmd {python_var}` broke on any path containing a space and threw away exit
  codes. Every command now runs through `subprocess` with real arguments.

**Things that are simply nicer**

- One voicebank builder instead of four (three of which the original labelled
  "legacy", "obsolete" and "even more obsolete").
- Pre-flight validation before binarizing: missing checkpoints, dictionaries that do
  not match your dataset's languages, pitch training with no MIDI in the data.
- A dataset summary after step 2 &mdash; speakers, segment counts, hours &mdash; so you find
  out about the singer whose files did not convert now, not 40 minutes into training.
- Binarized data goes to local disk by default; reading it over Drive's FUSE mount
  measurably slows training.
- Dictionaries are copied into your model folder, so the model folder is
  self-contained and still works in a fresh runtime.
- Interrupting the training cell also stops the training process, instead of leaving
  it holding the GPU so the next run dies with an out-of-memory error.

Credit where it is due: the original is by **usamireko**, maintained before that by
**MLo7Ghinsan**, and started life as **Kei's** notebook. The voicebank builder is
adapted from **agentasteriski's** DiffTrainer. This is a rewrite of their work, not a
replacement for it.